# Agentic Citation Retrieval Loop

**Workflow:**
1. LLM receives a legal query and generates 5 search queries
2. Each query runs through FAISS retriever and is reranked with CrossEncoder
3. Retrieved citations are compared against gold citations
4. Found gold citations kept in LLM context; missing ones silently dropped
5. LLM sees its full history (queries tried, what was found) and generates 5 NEW different queries
6. Loop repeats until all gold found OR max 10 iterations reached
7. Final report shows per-question, per-iteration breakdown

In [9]:
import json
import re
import time as _time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, List, Optional, Sequence, Union

import pandas as pd
import requests
from requests.exceptions import RequestException
from sentence_transformers import CrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
print('Imports OK')

/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Imports OK


## 2. Load models, vector store, and data

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = FAISS.load_local(
    '../faiss_index_new',
    embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 100},
)

# Change device to 'cpu' or 'cuda' as appropriate
reranker = CrossEncoder(
    '../models/cross_encoder_msmarco_finetuned',
    num_labels=1,
    device='mps',
)

train_csv = pd.read_csv('../data/train.csv')
laws_de   = pd.read_csv('../data/laws_de.csv')

print(f'train rows  : {len(train_csv)}')
print(f'laws_de rows: {len(laws_de)}')

/var/folders/gt/zsnjtvpd3891ny4sp9lsvh5c0000gn/T/ipykernel_469/1605583015.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at ../models/cross_encoder_msmarco_finetuned and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


train rows  : 1139
laws_de rows: 175933


## 3. System prompts

In [35]:
INITIAL_SYSTEM_PROMPT = (
    'You are a retrieval specialist for Swiss law.\n'
    'You will receive a legal case description (in German).\n'
    'Your ONLY task is to generate exactly 10 German search queries that will help\n'
    'retrieve the relevant statutory articles from a FAISS vector index.\n'
    '\n'
    'STRICT RULES:\n'
    '- DO NOT answer the legal question.\n'
    '- DO NOT suggest legal remedies or explain anything.\n'
    '- Queries must use statutory wording (as found in Swiss laws).\n'
    '- AVOID case-specific names, story details, or vague meta-words\n'
    '  like Rechtsbehelfe, Ansprueche, koennte, sollte.\n'
    '\n'
    'OUTPUT FORMAT (exactly 10 lines, no numbering, no symbols, no extra text):\n'
    '<query 1>\n'
    '<query 2>\n'
    '<query 3>\n'
    '<query 4>\n'
    '<query 5>\n'
    '<query 6>\n'
    '<query 7>\n'
    '<query 8>\n'
    '<query 9>\n'
    '<query 10>'
)

REFINE_SYSTEM_PROMPT = (
    'You are a retrieval specialist for Swiss law running an iterative search loop.\n'
    '\n'
    'In each iteration you:\n'
    '1. See the original legal case.\n'
    '2. See your previous search queries and how many gold citations each iteration found.\n'
    '3. See which gold citations have already been found (kept in context).\n'
    '4. Generate 5 NEW search queries that are DIFFERENT from all previous ones.\n'
    '   Focus on the legal concepts that are still MISSING.\n'
    '\n'
    'STRICT RULES:\n'
    '- DO NOT repeat queries that were already tried.\n'
    '- Queries must be short German keyword phrases using statutory wording.\n'
    '- DO NOT answer the question or explain anything.\n'
    '- Think about which legal areas or articles you have NOT explored yet.\n'
    '\n'
    'OUTPUT FORMAT (exactly 5 lines, no numbering, no symbols, no extra text):\n'
    '<query 1>\n'
    '<query 2>\n'
    '<query 3>\n'
    '<query 4>\n'
    '<query 5>'
)

print('Prompts defined.')

Prompts defined.


## 4. Core helpers

In [38]:
from langchain_core.messages import SystemMessage, HumanMessage

def generate_queries(prompt: str, system_prompt: str, temperature: float = 0.6) -> list:
    """Call Gemini and parse exactly 5 search queries from the response."""
    llm = ChatGoogleGenerativeAI(
        model='gemini-3-flash-preview',
        temperature=temperature,
    )
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    raw = response.content if isinstance(response.content, str) else response.content[0]['text']
    queries = [line.strip() for line in raw.strip().split('\n') if line.strip()]
    return queries  # safety cap


def retrieve_and_rerank(
    queries: list,
    retriever,
    reranker,
    gold: list | None = None,
    top_k_per_query: int = 10,
    verbose: bool = False,
) -> list:
    """
    For each query:
      1. Retrieve docs from FAISS.
      2. Print pre-rerank gold-hit stats, including gold-hit ranks in top-k.
      3. Rerank with CrossEncoder.
      4. Keep top_k_per_query and print post-rerank gold-hit ranks.
    Return a deduplicated list of citation strings.
    """
    all_citations = set()

    gold_set_lower = {g.lower() for g in gold} if gold else set()
    all_raw_citations = []

    # Use the same cut-off for pre- and post-rerank rank reporting for easy comparison.
    pre_top_k = top_k_per_query

    for i, q in enumerate(queries, start=1):
        docs = retriever.invoke(q)
        if not docs:
            if verbose:
                print(f"  [Pre-rerank] Query {i}: retrieved_docs=0, gold_hits=0")
            continue

        raw_citations = [str(doc.metadata.get('citation')) for doc in docs if doc.metadata.get('citation')]
        all_raw_citations.extend(raw_citations)

        if gold_set_lower and verbose:
            query_unique = set(raw_citations)
            query_gold_hits = [c for c in query_unique if c.lower() in gold_set_lower]

            pre_gold_rank_rows = []
            for rank, doc in enumerate(docs[:pre_top_k], start=1):
                cit = doc.metadata.get('citation')
                if cit and str(cit).lower() in gold_set_lower:
                    pre_gold_rank_rows.append((rank, str(cit)))

            pre_ranks = [r for r, _ in pre_gold_rank_rows]
            pre_ranked_citations = [c for _, c in pre_gold_rank_rows]

            print(
                f"  [Pre-rerank] Query {i}: retrieved_docs={len(docs)} "
                f"(target ~1000), unique_citations={len(query_unique)}, "
                f"gold_hits={len(query_gold_hits)}, "
                f"gold_ranks_top{pre_top_k}={pre_ranks if pre_ranks else 'NONE'}"
            )
            if pre_ranked_citations:
                print(f"    [Pre-rerank] Gold citations @ ranks: {pre_ranked_citations}")

        pairs = [(q, doc.page_content) for doc in docs]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

        if gold_set_lower and verbose:
            post_gold_rank_rows = []
            for rank, (doc, _) in enumerate(ranked[:top_k_per_query], start=1):
                cit = doc.metadata.get('citation')
                if cit and str(cit).lower() in gold_set_lower:
                    post_gold_rank_rows.append((rank, str(cit)))

            post_ranks = [r for r, _ in post_gold_rank_rows]
            post_ranked_citations = [c for _, c in post_gold_rank_rows]
            print(
                f"  [Post-rerank] Query {i}: kept_top{top_k_per_query}, "
                f"gold_ranks_top{top_k_per_query}={post_ranks if post_ranks else 'NONE'}"
            )
            if post_ranked_citations:
                print(f"    [Post-rerank] Gold citations @ ranks: {post_ranked_citations}")

        for doc, _ in ranked[:top_k_per_query]:
            cit = doc.metadata.get('citation')
            if cit:
                all_citations.add(str(cit))

    if gold_set_lower and verbose:
        all_unique = set(all_raw_citations)
        total_gold_hits = [c for c in all_unique if c.lower() in gold_set_lower]
        print(
            f"  [Pre-rerank][TOTAL] queries={len(queries)}, "
            f"retrieved_docs_total={len(all_raw_citations)} (target ~{len(queries) * 1000}), "
            f"unique_citations_total={len(all_unique)}, "
            f"gold_hits_total={len(total_gold_hits)}"
        )

    return list(all_citations)


def compare_citations(retrieved: list, gold: list):
    """Returns (found, missing) where found = gold citations present in retrieved."""
    retrieved_lower = {c.lower() for c in retrieved}
    found   = [g for g in gold if g.lower() in retrieved_lower]
    missing = [g for g in gold if g.lower() not in retrieved_lower]
    return found, missing


def build_refinement_prompt(original_query: str, history: list, found_so_far: list, missing_so_far: list) -> str:
    """
    Build the user-turn message for refinement calls.
    The LLM sees full history so it knows exactly what it already tried.
    """
    lines = []
    lines.append('=== LEGAL CASE ===')
    lines.append(original_query)
    lines.append('')
    lines.append('=== SEARCH HISTORY ===')
    for entry in history:
        lines.append(f"--- Iteration {entry['iteration']} ---")
        lines.append('Queries used:')
        for q in entry['queries']:
            lines.append(f'  - {q}')
        lines.append(
            f"Gold citations found this iteration: {entry['found']}" if entry['found']
            else 'Gold citations found this iteration: NONE'
        )
        lines.append('')
    lines.append('=== CITATIONS FOUND SO FAR ===')
    lines.append(str(found_so_far) if found_so_far else 'None yet')
    lines.append('')
    lines.append('=== CITATIONS STILL MISSING ===')
    lines.append(str(missing_so_far) if missing_so_far else 'All found!')
    lines.append('')
    lines.append(
        'Generate 5 new search queries that target the MISSING citations '
        'and do NOT repeat any query already tried.'
    )
    return '\n'.join(lines)


print('Helpers defined.')

Helpers defined.


## 5. Data classes

In [12]:
@dataclass
class IterationRecord:
    iteration:        int
    queries:          list    # search queries used this iteration
    retrieved:        list    # all citation IDs retrieved
    new_found:        list    # gold citations newly discovered this iteration
    cumulative_found: list    # total gold found up to and including this iteration
    missing:          list    # gold still missing after this iteration


@dataclass
class QuestionResult:
    question_idx:  int
    query:         str
    gold:          list
    iterations:    list = field(default_factory=list)
    final_found:   list = field(default_factory=list)
    final_missing: list = field(default_factory=list)
    total_iters:   int  = 0


print('Dataclasses defined.')

Dataclasses defined.


### QWEN

In [117]:
## mistral model
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.language_models.chat_models import  BaseChatModel # to create our own class
from langchain_core.messages import (
    BaseMessage, AIMessage, SystemMessage, HumanMessage, ToolMessage
)
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import BaseTool
from langchain_core.utils.function_calling import convert_to_openai_tool
import json
import time as _time

import pandas as pd
import requests
from requests.exceptions import RequestException
import uuid
from typing import Any, List, Optional, Sequence, Union, Callable
# tool config
TOOL_RESULT_MAX_CHARS = 2000
HISTORY_WINDOW = 8 # how many last messages to keep in context

# helper functions

def _merge_consecutive_roles(messages: List[dict]) -> List[dict]:
    """
    Mistral chat template requires strict user/assistant alternation.
    Merge any consecutive messages with the same role into one,
    joining their content with a newline separator.
    System messages are always kept at the front and not merged.
    """

    system_msg = [m for m in messages if m["role"] == "system"]
    non_system = [m for m in messages if m["role"] != "system"]
    merged: List[dict] = []

    for msg in non_system:
        if merged and merged[-1]["role"] == msg["role"]:
            merged[-1] = {
                "role": msg["role"],
                "content":merged[-1]["content"] + "\n\n" + msg["content"],
            }
        else:
            merged.append(msg)

    return system_msg + merged

def _parse_tool_call(text: str) -> tuple[list, str]:
    """
    Extract tool-call JSON from model output.
    Handles bare JSON and markdown-fenced ```json ... ``` blocks.
    Also handles escaped underscores (e.g. retriever\\_tool → retriever_tool).
    """
    # Unescape markdown-escaped underscores the model sometimes emits
    text_clean = text.replace("\\_", "_")

    candidates: list[tuple[str, str]] = []

    for m in re.finditer(r"```(?:json)?\s*(.*?)```", text_clean, re.DOTALL):
        candidates.append((m.group(1).strip(), m.group(0)))

    for m in re.finditer(r'\{[^{}]*"tool"\s*:[^{}]*\{[^{}]*\}[^{}]*\}', text_clean, re.DOTALL):
        candidates.append((m.group(0).strip(), m.group(0)))

    for raw, full_match in candidates:
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if "tool" not in parsed:
            continue
        args = parsed.get("tool_input") or parsed.get("arguments") or {}
        tool_calls = [{
            "name": parsed["tool"],
            "args": args,
            "id":   f"call_{uuid.uuid4().hex[:8]}",
            "type": "tool_call",
        }]
        return tool_calls, text_clean.replace(full_match, "").strip()

    return [], text

### qwen 2.5 7b
class ModalQwenChat(BaseChatModel):
    api_url: str
    temperature: float = 0.1
    max_new_tokens: int = 512
    top_p: float = 0.9
    system_prompt: str
    bound_tools: List[dict] = []
    max_retries: int = 3
    http_timeout: int = 600
    api_messages: List[Any] = []

    @property
    def _llm_type(self) -> str:
        return "modal_qwen_chat"

    def bind_tools(
        self,
        tools: Sequence[Union[dict, type, Callable, BaseTool]],
        **kwargs: Any
    ) -> "ModalQwenChat":
        tool_schemas = [convert_to_openai_tool(t)["function"] for t in tools]
        return self.model_copy(update={"bound_tools": tool_schemas})

    def _content_to_text(self, content: Any) -> str:
        if isinstance(content, str):
            return content
        if content is None:
            return ""
        try:
            return json.dumps(content, ensure_ascii=False)
        except Exception:
            return str(content)

    def _base_message_to_api(self, msg: BaseMessage) -> dict:
        if isinstance(msg, SystemMessage):
            return {"role": "system", "content": self._content_to_text(msg.content)}

        if isinstance(msg, HumanMessage):
            return {"role": "user", "content": self._content_to_text(msg.content)}

        if isinstance(msg, ToolMessage):
            out = {"role": "tool", "content": self._content_to_text(msg.content)}
            if hasattr(msg, "tool_call_id") and msg.tool_call_id:
                out["tool_call_id"] = msg.tool_call_id
            return out

        if isinstance(msg, AIMessage):
            return {"role": "assistant", "content": self._content_to_text(msg.content)}

        return {"role": "user", "content": self._content_to_text(getattr(msg, "content", str(msg)))}

    def _coerce_api_messages(self, raw_messages: Sequence[Any]) -> List[dict]:
        cooked: List[dict] = []
        for item in raw_messages:
            if isinstance(item, dict):
                role = str(item.get("role", "user")).strip() or "user"
                msg = {
                    "role": role,
                    "content": self._content_to_text(item.get("content", "")),
                }
                if "tool_call_id" in item:
                    msg["tool_call_id"] = item["tool_call_id"]
                cooked.append(msg)
                continue

            if isinstance(item, BaseMessage):
                cooked.append(self._base_message_to_api(item))
                continue

            cooked.append({"role": "user", "content": self._content_to_text(item)})

        return cooked

    def _build_messages(self, messages: List[BaseMessage]) -> List[dict]:
        """Build API messages without collapsing user content into system content."""
        body: List[dict] = []

        has_system = any(isinstance(m, SystemMessage) for m in messages)
        if not has_system and self.system_prompt.strip():
            body.append({"role": "system", "content": self.system_prompt})

        for msg in messages:
            body.append(self._base_message_to_api(msg))

        system_part = [m for m in body if m["role"] == "system"]
        non_system = [m for m in body if m["role"] != "system"]
        windowed = non_system[-HISTORY_WINDOW:] if HISTORY_WINDOW > 0 else non_system

        return system_part + windowed

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> ChatResult:

        override_api_messages = kwargs.pop("api_messages", None)

        if override_api_messages is not None:
            api_messages = self._coerce_api_messages(override_api_messages)
        elif self.api_messages:
            api_messages = self._coerce_api_messages(self.api_messages)
        else:
            api_messages = self._build_messages(messages)

        if not api_messages and self.system_prompt.strip():
            api_messages = [{"role": "system", "content": self.system_prompt}]

        # Keep last payload for debug visibility.
        self.api_messages = api_messages

        print("api messages:", api_messages)
        total_chars = sum(len(m.get("content", "")) for m in api_messages)
        est_tokens  = total_chars // 3
        print(f"→ POST to Modal (Qwen) | {len(api_messages)} msgs | {total_chars:,} chars | ~{est_tokens:,} tokens")

        payload = {
            "messages": api_messages,
            "max_new_tokens": self.max_new_tokens,
            "temperature": self.temperature,
            "top_p": self.top_p,
        }

        last_err = None

        for attempt in range(1, self.max_retries + 1):
            try:
                response = requests.post(
                    self.api_url,
                    json=payload,
                    timeout=self.http_timeout,
                )

                if response.ok:
                    break

                try:
                    err_body = response.json()
                except Exception:
                    err_body = response.text[:500]

                print(f"[ModalQwenChat] HTTP {response.status_code} (attempt {attempt}/{self.max_retries}) — {err_body}")
                last_err = response

                if attempt < self.max_retries:
                    wait = 2 ** attempt
                    print(f"⏳ Retrying in {wait}s…")
                    _time.sleep(wait)

            except RequestException as e:
                print(f"[ModalQwenChat] Network error (attempt {attempt}/{self.max_retries}) — {e}")
                last_err = e

                if attempt < self.max_retries:
                    wait = 2 ** attempt
                    print(f"⏳ Retrying in {wait}s…")
                    _time.sleep(wait)

        if isinstance(last_err, RequestException):
            raise last_err

        if last_err is not None and hasattr(last_err, "raise_for_status"):
            if not last_err.ok:
                last_err.raise_for_status()

        text: str = response.json()["response"].strip()

        if stop:
            for s in stop:
                text = text.split(s)[0]

        tool_calls, text = _parse_tool_call(text) if self.bound_tools else ([], text)

        return ChatResult(
            generations=[
                ChatGeneration(
                    message=AIMessage(content=text, tool_calls=tool_calls)
                )
            ]
        )



In [118]:
llm = ModalQwenChat(
    api_url="https://keshavsharma25--completion.modal.run",
    temperature=0.5,
    max_new_tokens=512,
    top_p=0.95,
    max_retries=1,
    http_timeout=600,
    system_prompt="you are a helpful assistant that answers questions in english",
)
llm.invoke("who are you?")

api messages: [{'role': 'system', 'content': 'you are a helpful assistant that answers questions in english'}, {'role': 'user', 'content': 'who are you?'}]
→ POST to Modal (Qwen) | 2 msgs | 73 chars | ~24 tokens


AIMessage(content=".imgur.com\n\nI'm an AI assistant created by Anthropic to be helpful, harmless, and honest. I don't have physical form or a website at imgur.com. How can I assist you today?", additional_kwargs={}, response_metadata={}, id='lc_run--019d6027-e296-7e60-9a4a-eb35a34e44f1-0', tool_calls=[], invalid_tool_calls=[])

## 6. Agentic loop

In [ ]:
def run_agentic_loop(
    question_idx:    int,
    retriever,
    reranker,
    max_iterations:  int  = 10,
    top_k_per_query: int  = 10,
    verbose:         bool = True,
) -> QuestionResult:
    """
    Agentic citation-retrieval loop for one question.


    The LLM:
      - Generates search queries
      - Learns which gold citations it found (these stay in context)
      - Knows which are still missing
      - Generates NEW, different queries each iteration targeting missing ones
    Stops when all gold found OR max_iterations reached.
    """
    original_query = train_csv.iloc[question_idx]['query']
    gold_citations = [
        c.strip()
        for c in str(train_csv.iloc[question_idx]['gold_citations']).split(';')
        if c.strip()
    ]

    result = QuestionResult(
        question_idx=question_idx,
        query=original_query,
        gold=gold_citations,
    )

    found_so_far:   list = []
    missing_so_far: list = list(gold_citations)
    history:        list = []   # full history that LLM receives each iteration

    if verbose:
        print('\n' + '=' * 72)
        print(f'  QUESTION {question_idx}')
        print(f'  Query   : {original_query[:110]}...')
        print(f'  Gold ({len(gold_citations)}): {gold_citations}')
        print('=' * 72)

    for iteration in range(1, max_iterations + 1):
        if verbose:
            print(f'\n-- Iteration {iteration}/{max_iterations} --')

        # ── Generate queries ──────────────────────────────────────────────
        if iteration == 1:
            prompt        = original_query
            system_prompt = INITIAL_SYSTEM_PROMPT
        else:
            prompt        = build_refinement_prompt(original_query, history, found_so_far, missing_so_far)
            system_prompt = REFINE_SYSTEM_PROMPT

        # Slightly increase temperature each iteration to explore more diverse queries
        temp    = min(0.6 + (iteration - 1) * 0.04, 1.0)
        queries = generate_queries(prompt, system_prompt, temperature=temp)

        if verbose:
            print(f'  Generated {len(queries)} queries (temp={temp:.2f}):')
            for q in queries:
                print(f'    * {q}')

        # ── Retrieve + rerank ─────────────────────────────────────────────
        retrieved = retrieve_and_rerank(
            queries=queries,
            retriever=retriever,
            reranker=reranker,
            gold=gold_citations,
            top_k_per_query=top_k_per_query,
            verbose=verbose,
        )
        if verbose:
            print(f'  Retrieved {len(retrieved)} unique citations after rerank')

        # ── Compare against gold ──────────────────────────────────────────
        iter_found, _ = compare_citations(retrieved, gold_citations)
        new_this_iter  = [c for c in iter_found if c not in found_so_far]
        found_so_far   = list(set(found_so_far + new_this_iter))
        missing_so_far = [g for g in gold_citations if g not in found_so_far]

        if verbose:
            print(f'  New gold found this iter  : {new_this_iter if new_this_iter else "NONE"}')
            print(f'  Cumulative found          : {found_so_far}  [{len(found_so_far)}/{len(gold_citations)}]')
            print(f'  Still missing             : {missing_so_far}')

        # ── Record ───────────────────────────────────────────────────────
        record = IterationRecord(
            iteration=iteration,
            queries=queries,
            retrieved=retrieved,
            new_found=new_this_iter,
            cumulative_found=list(found_so_far),
            missing=list(missing_so_far),
        )
        result.iterations.append(record)

        # Update history for next iteration's LLM context
        history.append({'iteration': iteration, 'queries': queries, 'found': new_this_iter})

        # ── Early exit if all gold found ──────────────────────────────────
        if not missing_so_far:
            if verbose:
                print(f'\n  ALL {len(gold_citations)} gold citations found at iteration {iteration}!')
            break

    result.final_found   = found_so_far
    result.final_missing = missing_so_far
    result.total_iters   = len(result.iterations)
    return result


print('run_agentic_loop() defined.')

run_agentic_loop() defined.


## 7. Reporting helpers

In [13]:
def print_question_report(res: QuestionResult) -> None:
    total_gold  = len(res.gold)
    total_found = len(res.final_found)
    recall = total_found / total_gold if total_gold else 0.0

    print('\n' + '=' * 72)
    print(f'  REPORT -- Question {res.question_idx}')
    print('=' * 72)
    print(f'  Query  : {res.query[:100]}...')
    print(f'  Gold citations ({total_gold}): {res.gold}')
    print(f'  Iterations run : {res.total_iters}')
    print(f'  Final recall   : {total_found}/{total_gold}  ({recall:.1%})')
    print(f'  Found   : {res.final_found}')
    print(f'  Missing : {res.final_missing}')
    print()
    print('  Iteration breakdown:')
    print(f"  {'Iter':>4}  {'#Queries':>8}  {'Retrieved':>9}  {'NewFound':>8}  {'CumulFound':>10}")
    print('  ' + '-' * 55)
    for ir in res.iterations:
        print(
            f'  {ir.iteration:>4}  '
            f'{len(ir.queries):>8}  '
            f'{len(ir.retrieved):>9}  '
            f'{len(ir.new_found):>8}  '
            f'{len(ir.cumulative_found):>10}/{total_gold}'
        )
    print('=' * 72)


def aggregate_report(results: list) -> pd.DataFrame:
    rows = []
    for r in results:
        total_gold  = len(r.gold)
        total_found = len(r.final_found)
        rows.append({
            'question_idx':      r.question_idx,
            'query_snippet':     r.query[:60] + '...',
            'gold_total':        total_gold,
            'found':             total_found,
            'missing':           len(r.final_missing),
            'recall':            round(total_found / total_gold, 3) if total_gold else 0.0,
            'iterations_run':    r.total_iters,
            'found_citations':   ';'.join(r.final_found),
            'missing_citations': ';'.join(r.final_missing),
        })
    return pd.DataFrame(rows)


print('Reporting helpers defined.')

Reporting helpers defined.


## 8. Smoke test — single question

In [28]:
test_result = run_agentic_loop(
    question_idx=1,
    retriever=retriever,
    reranker=reranker,
    max_iterations=10,
    top_k_per_query=10,
    verbose=True,
)

print_question_report(test_result)


  QUESTION 0
  Query   : Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunterne...
  Gold (3): ['Art. 10a Abs. 1 USG', 'Art. 2 Abs. 1 UVPV', 'Art. 10a Abs. 1 UVG']

-- Iteration 1/10 --
  Generated 5 queries (temp=0.60):
    * wesentliche Änderung
    * erhebliche Auswirkungen auf die Umwelt
    * Anhang UVPV
    * Lärmimmissionen
    * Grenzwerte überschritten
  [Pre-rerank] Query 1: retrieved_docs=100 (target ~1000), unique_citations=100, gold_hits=0
  [Pre-rerank] Query 2: retrieved_docs=100 (target ~1000), unique_citations=100, gold_hits=0
  [Pre-rerank] Query 3: retrieved_docs=100 (target ~1000), unique_citations=100, gold_hits=0
  [Pre-rerank] Query 4: retrieved_docs=100 (target ~1000), unique_citations=100, gold_hits=0
  [Pre-rerank] Query 5: retrieved_docs=100 (target ~1000), unique_citations=100, gold_hits=0
  [Pre-rerank][TOTAL] queries=5, retrieved_docs_total=500 (target ~5000), unique_citations_total=476, gold_hits_to

KeyboardInterrupt: 

## 9. Run across multiple questions

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
QUESTIONS_TO_RUN  = list(range(0, 10))   # adjust range as needed
MAX_ITERATIONS    = 10
TOP_K_PER_QUERY   = 10

all_results = []

for idx in QUESTIONS_TO_RUN:
    res = run_agentic_loop(
        question_idx=idx,
        retriever=retriever,
        reranker=reranker,
        max_iterations=MAX_ITERATIONS,
        top_k_per_query=TOP_K_PER_QUERY,
        verbose=True,
    )
    all_results.append(res)

print(f'\nDone -- processed {len(all_results)} questions.')

## 10. Per-question reports

In [ ]:
for res in all_results:
    print_question_report(res)

## 11. Aggregate summary table

In [ ]:
summary_df = aggregate_report(all_results)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)
display(summary_df)

mean_recall = summary_df['recall'].mean()
perfect     = (summary_df['recall'] == 1.0).sum()
print(f'\nMean recall across {len(summary_df)} questions : {mean_recall:.1%}')
print(f'Questions with perfect recall (1.0)           : {perfect}/{len(summary_df)}')
print(f'Mean iterations used                          : {summary_df["iterations_run"].mean():.1f}')

## 12. Save results to CSV

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

summary_df.to_csv('../results/agentic_retrieval_results.csv', index=False)
print('Summary saved to ../results/agentic_retrieval_results.csv')

# Detailed iteration-level log
iter_rows = []
for res in all_results:
    for ir in res.iterations:
        iter_rows.append({
            'question_idx':    res.question_idx,
            'iteration':       ir.iteration,
            'queries':         ' | '.join(ir.queries),
            'retrieved_count': len(ir.retrieved),
            'new_found':       ';'.join(ir.new_found),
            'cumulative_found':';'.join(ir.cumulative_found),
            'missing':         ';'.join(ir.missing),
        })

iter_df = pd.DataFrame(iter_rows)
iter_df.to_csv('../results/agentic_retrieval_iterations.csv', index=False)
print('Iteration log saved to ../results/agentic_retrieval_iterations.csv')
display(iter_df.head(20))

In [21]:
# For each question in train:
# Positive pairs: (query, gold_article_text)  → label 1.0
# Negative pairs: (query, retrieved_non_gold_text) → label 0.0

from sentence_transformers import InputExample

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 100},
)

train_samples = []
count = 0
for _, row in train_csv.iterrows():
    
    query = row['query']
    gold_cits = row['gold_citations'].split(';')
    
    # positives
    for cit in gold_cits:
        text = laws_de[laws_de['citation'] == cit.strip()]['text'].values
        if len(text):
            train_samples.append(InputExample(texts=[query, text[0]], label=1.0))
    
    # negatives — retrieved but not gold (hard negatives are best)
    # use your FAISS retriever to get top-100, exclude golds
    retrieved = retriever.invoke(query)
    non_gold = [d for d in retrieved if d.metadata['citation'] not in gold_cits]
    for doc in non_gold[:len(gold_cits) * 3]:  # 3:1 negative ratio
        train_samples.append(InputExample(texts=[query, doc.page_content], label=0.0))
    count += 1
    if count % 10 == 0:
        print(f'Processed {count}/{len(train_csv)} questions, total samples so far: {len(train_samples)}')

Processed 10/1139 questions, total samples so far: 178
Processed 20/1139 questions, total samples so far: 347
Processed 30/1139 questions, total samples so far: 479
Processed 40/1139 questions, total samples so far: 604
Processed 50/1139 questions, total samples so far: 850
Processed 60/1139 questions, total samples so far: 1032
Processed 70/1139 questions, total samples so far: 1207
Processed 80/1139 questions, total samples so far: 1311
Processed 90/1139 questions, total samples so far: 1474
Processed 100/1139 questions, total samples so far: 1590
Processed 110/1139 questions, total samples so far: 1675
Processed 120/1139 questions, total samples so far: 1905
Processed 130/1139 questions, total samples so far: 2054
Processed 140/1139 questions, total samples so far: 2194
Processed 150/1139 questions, total samples so far: 2283
Processed 160/1139 questions, total samples so far: 2379
Processed 170/1139 questions, total samples so far: 2526
Processed 180/1139 questions, total samples s

In [24]:
# save train_samples to disk for later use
import pickle
with open('../data/cross_encoder_training_samples.pkl', 'wb') as f:
    pickle.dump(train_samples, f)
print(f'Training samples saved to ../data/cross_encoder_training_samples.pkl, total samples: {len(train_samples)}')

Training samples saved to ../results/cross_encoder_training_samples.pkl, total samples: 17158


In [25]:
pos = sum(1 for s in train_samples if s.label == 1.0)
neg = sum(1 for s in train_samples if s.label == 0.0)
print(f"Positives: {pos}, Negatives: {neg}, Ratio: {neg/pos:.1f}:1")

Positives: 3262, Negatives: 13896, Ratio: 4.3:1


In [ ]:
def generate_queries(prompt: str, system_prompt: str, temperature: float = 0.6) -> list:
    """Call Gemini and parse exactly 5 search queries from the response."""
    llm = ChatGoogleGenerativeAI(
        model='gemini-3-flash-preview',
        temperature=temperature,
    )
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    raw = response.content if isinstance(response.content, str) else response.content[0]['text']
    queries = [line.strip() for line in raw.strip().split('\n') if line.strip()]
    return queries  # safety cap

In [53]:
TEST_PROMPT = """You are a Swiss legal retrieval assistant.

You will receive a legal case description.

Your task is to generate exactly 10 short search queries in German to retrieve relevant Swiss law texts.

GOAL:
The queries should match the wording of statutes (laws), not case descriptions.

REQUIREMENTS:

1. Extract the main legal concept(s) from the case.

2. Generate queries that cover DIFFERENT legal aspects:

   * the core legal concept
   * conditions / requirements
   * liability or consequences
   * procedural aspects (e.g., complaints, actions)
   * general legal formulation (broad wording)

3. Each query must:

   * be short (keywords, not sentences)
   * use legal/statutory language
   * resemble wording used in laws

STRICTLY AVOID:

* case-specific facts (names, objects, locations)
* storytelling or explanations
* guessing article numbers

OUTPUT FORMAT:

* Exactly 10 lines
* Each line = one query
* No numbering
* No extra text
"""

In [70]:
gold_from_train_idx = 123   # train row used for gold citations
import pandas as pd
train_csv = pd.read_csv('../data/train.csv')
query_questions = generate_queries(train_csv.iloc[gold_from_train_idx]['query'], TEST_PROMPT, temperature=0)
print(query_questions)
# 1 --> 
# 2 --> [(3, 'Art. 974 Abs. 2 ZGB'), (5, 'Art. 976 ZGB'), (81, 'Art. 661 ZGB')]

['Schutz der Persönlichkeit des Arbeitnehmers', 'Überwachungs- und Kontrollsysteme Gesundheitsschutz', 'Bearbeitung von Personaldaten im Arbeitsverhältnis', 'Fürsorgepflicht des Arbeitgebers Persönlichkeitsschutz', 'Verhaltensüberwachung am Arbeitsplatz', 'Datenschutz am Arbeitsplatz', 'Schutz der Privatsphäre Arbeitnehmer', 'Zulässigkeit von Überwachungsmassnahmen', 'Gesundheitsschutz am Arbeitsplatz Überwachung', 'Persönlichkeitsrechte im Arbeitsrecht']


In [71]:
# Pre-rerank focused analyzer for a list of generated queries.
# Prints only: gold citation names + first pre-rerank rank per query,
# and an aggregate contribution summary across all queries.

# -------------------------------
# 1) USER INPUTS
# -------------------------------

retriever_k = 1000        # pre-rerank depth to inspect

# -------------------------------
# 2) HELPERS
# -------------------------------
def _parse_gold_from_train(train_df: pd.DataFrame, idx: int) -> list[str]:
    if idx < 0 or idx >= len(train_df):
        raise IndexError(f"gold_from_train_idx={idx} out of range [0, {len(train_df)-1}]")
    raw = str(train_df.iloc[idx]["gold_citations"])
    return [c.strip() for c in raw.split(";") if c.strip()]


def analyze_pre_rerank_query(
    query_text: str,
    eval_gold: list[str],
    retriever_k: int = 1000,
) -> dict:
    docs = retriever.invoke(query_text)[:retriever_k]

    # First rank seen for each citation (case-insensitive key).
    first_rank_by_lower = {}
    first_citation_by_lower = {}
    for rank, d in enumerate(docs, start=1):
        cit = d.metadata.get("citation")
        if not cit:
            continue
        cit_str = str(cit).strip()
        if not cit_str:
            continue
        key = cit_str.lower()
        if key not in first_rank_by_lower:
            first_rank_by_lower[key] = rank
            first_citation_by_lower[key] = cit_str

    # Gold matches with ranks in gold order.
    matched_gold_with_ranks = []
    for g in eval_gold:
        key = g.lower()
        if key in first_rank_by_lower:
            matched_gold_with_ranks.append((g, first_rank_by_lower[key]))

    return {
        "query": query_text,
        "retrieved_docs": len(docs),
        "unique_citations": len(first_rank_by_lower),
        "all_unique_citation_lowers": set(first_rank_by_lower.keys()),
        "matched_gold_with_ranks": matched_gold_with_ranks,
        "matched_gold_lower": {g.lower() for g, _ in matched_gold_with_ranks},
    }


# -------------------------------
# 3) RUN PRE-RERANK REPORT
# -------------------------------
if "query_questions" not in globals():
    raise ValueError("query_questions is not defined. Run the query-generation cell first.")

queries_to_run = query_questions if isinstance(query_questions, list) else [query_questions]
queries_to_run = [q for q in queries_to_run if isinstance(q, str) and q.strip()]

if not queries_to_run:
    raise ValueError("No valid queries found in query_questions.")

eval_gold = _parse_gold_from_train(train_csv, gold_from_train_idx)
gold_lookup = {g.lower(): g for g in eval_gold}

print("\n" + "=" * 96)
print("PRE-RERANK QUERY REPORT (GOLD CITATIONS + RANKS)")
print("=" * 96)
print(f"Gold source train index : {gold_from_train_idx}")
print(f"Gold citation count     : {len(eval_gold)}")
print(f"Queries to evaluate     : {len(queries_to_run)}")
print(f"Retriever top-k         : {retriever_k}")

results = []
all_unique_citations_lower = set()
cumulative_gold_found_lower = set()

for i, q in enumerate(queries_to_run, start=1):
    rep = analyze_pre_rerank_query(
        query_text=q,
        eval_gold=eval_gold,
        retriever_k=retriever_k,
    )

    # Track unique citations across all queries.
    all_unique_citations_lower |= rep["all_unique_citation_lowers"]

    matched_now = rep["matched_gold_lower"]
    new_contrib_lower = matched_now - cumulative_gold_found_lower
    cumulative_gold_found_lower |= matched_now

    rep["query_idx"] = i
    rep["new_gold_contrib_count"] = len(new_contrib_lower)
    rep["new_gold_contrib_citations"] = [
        g for g in eval_gold if g.lower() in new_contrib_lower
    ]
    results.append(rep)

    print("\n" + "-" * 96)
    print(f"Query {i}: {q}")
    print(f"Gold matches (name, pre-rank): {rep['matched_gold_with_ranks'] if rep['matched_gold_with_ranks'] else 'NONE'}")
    print(
        f"Stats: matched={len(rep['matched_gold_with_ranks'])}/{len(eval_gold)} | "
        f"new_gold_contrib={rep['new_gold_contrib_count']}"
    )

final_gold_found = len(cumulative_gold_found_lower)
final_recall = (final_gold_found / len(eval_gold)) if eval_gold else 0.0

print("\n" + "=" * 96)
print(
    f"Using these {len(queries_to_run)} queries, total unique pre-rerank citations generated: "
    f"{len(all_unique_citations_lower)}"
)
print(f"Final pre-rerank recall: {final_gold_found}/{len(eval_gold)} ({final_recall:.2%})")
print("Per-query gold contribution summary:")
for i, rep in enumerate(results, start=1):
    print(
        f"  Query {i}: matched={len(rep['matched_gold_with_ranks'])} | "
        f"new_gold_contrib={rep['new_gold_contrib_count']}"
    )

top_queries = sorted(
    results,
    key=lambda r: (
        r["new_gold_contrib_count"],
        len(r["matched_gold_with_ranks"]),
        -r["query_idx"],
    ),
    reverse=True,
)[:5]

print("Top 5 queries by contribution (sorted):")
for rank, rep in enumerate(top_queries, start=1):
    print(
        f"  {rank}. Query {rep['query_idx']}: "
        f"new_gold_contrib={rep['new_gold_contrib_count']}, "
        f"matched={len(rep['matched_gold_with_ranks'])}/{len(eval_gold)}"
    )
    print(f"     text: {rep['query']}")

print("\nDone. Detailed per-query objects are saved in `results`.")


PRE-RERANK QUERY REPORT (GOLD CITATIONS + RANKS)
Gold source train index : 123
Gold citation count     : 4
Queries to evaluate     : 10
Retriever top-k         : 1000

------------------------------------------------------------------------------------------------
Query 1: Schutz der Persönlichkeit des Arbeitnehmers
Gold matches (name, pre-rank): NONE
Stats: matched=0/4 | new_gold_contrib=0

------------------------------------------------------------------------------------------------
Query 2: Überwachungs- und Kontrollsysteme Gesundheitsschutz
Gold matches (name, pre-rank): NONE
Stats: matched=0/4 | new_gold_contrib=0

------------------------------------------------------------------------------------------------
Query 3: Bearbeitung von Personaldaten im Arbeitsverhältnis
Gold matches (name, pre-rank): [('Art. 328b OR', 24)]
Stats: matched=1/4 | new_gold_contrib=1

------------------------------------------------------------------------------------------------
Query 4: Fürsorgepfl

# Experiment: total citation frequency across all retrieved docs from query_questions.
# We retrieve top-10000 per query, count TOTAL occurrences, and inspect top-100 citations.
# Force full IVF search by setting nprobe = nlist (when using an IVF index).

from collections import Counter

TOP_K_COMMON = 10000

if "query_questions" not in globals():
    raise ValueError("query_questions is not defined. Run the query-generation cell first.")

if "vectorstore" not in globals():
    raise ValueError("vectorstore is not defined. Run the model/vectorstore loading cell first.")

queries_to_run = query_questions if isinstance(query_questions, list) else [query_questions]
queries_to_run = [q.strip() for q in queries_to_run if isinstance(q, str) and q.strip()]

if not queries_to_run:
    raise ValueError("No valid queries found in query_questions.")

if "gold_from_train_idx" not in globals():
    gold_from_train_idx = 1

raw_gold = str(train_csv.iloc[gold_from_train_idx]["gold_citations"])
gold_citations = [c.strip() for c in raw_gold.split(";") if c.strip()]
gold_lower = {g.lower() for g in gold_citations}

# Enforce exhaustive IVF probing if available.
index_type = type(vectorstore.index).__name__
if hasattr(vectorstore.index, "nprobe") and hasattr(vectorstore.index, "nlist"):
    vectorstore.index.nprobe = vectorstore.index.nlist

effective_nprobe = getattr(vectorstore.index, "nprobe", None)
effective_nlist = getattr(vectorstore.index, "nlist", None)

print("\n" + "=" * 100)
print("TOTAL OCCURRENCE FREQUENCY EXPERIMENT (TOP-10000 PER QUERY)")
print("=" * 100)
print(f"Queries evaluated     : {len(queries_to_run)}")
print(f"Top-k per query       : {TOP_K_COMMON}")
print(f"Expected max rows     : {len(queries_to_run) * TOP_K_COMMON}")
print(f"Gold source train idx : {gold_from_train_idx}")
print(f"Gold citation count   : {len(gold_citations)}")
print(f"Index type            : {index_type}")
if effective_nprobe is not None and effective_nlist is not None:
    print(f"IVF probing           : nprobe={effective_nprobe}, nlist={effective_nlist} (full search)")

# Count total occurrences across all retrieved docs.
occurrence_counter = Counter()
representative_text = {}
total_rows_retrieved = 0

for i, q in enumerate(queries_to_run, start=1):
    docs = vectorstore.similarity_search(q, k=TOP_K_COMMON)
    total_rows_retrieved += len(docs)

    for d in docs:
        cit = d.metadata.get("citation")
        if not cit:
            continue
        cit_str = str(cit).strip()
        if not cit_str:
            continue

        key = cit_str.lower()
        occurrence_counter[key] += 1
        if key not in representative_text:
            representative_text[key] = cit_str

    print(f"Query {i}: retrieved_docs={len(docs)}")

if not occurrence_counter:
    raise ValueError("No citations found in retrieval results.")

# Build full frequency table.
freq_rows = []
for key, occ in occurrence_counter.items():
    freq_rows.append(
        {
            "citation": representative_text[key],
            "total_freq": int(occ),
            "is_gold": key in gold_lower,
        }
    )

freq_df = pd.DataFrame(freq_rows).sort_values("total_freq", ascending=False).reset_index(drop=True)

# Top-100 table requested.
top100_freq_df = freq_df.head(100).copy()
top100_freq_df.insert(0, "rank", range(1, len(top100_freq_df) + 1))

top100_gold_hits = int(top100_freq_df["is_gold"].sum())

print("\n" + "-" * 100)
print(f"Total rows actually retrieved : {total_rows_retrieved}")
print(f"Total unique citations        : {len(freq_df)}")
print(f"Gold in top-100 by total freq : {top100_gold_hits}/{len(gold_citations)}")

print("\nTop 100 citations by TOTAL frequency (with gold flag):")
display(top100_freq_df)

print("\nDone. DataFrames available: freq_df, top100_freq_df")

#### lets try to generate data for finetuning model (generating german questions given english query)

In [ ]:
def generate_queries(prompt: str, system_prompt: str, temperature: float = 0.6) -> list:
    """Call Gemini and parse exactly 5 search queries from the response."""
    llm = ChatGoogleGenerativeAI(
        model='gemini-2.0-flash',
        temperature=temperature,
    )
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    raw = response.content if isinstance(response.content, str) else response.content[0]['text']
    queries = [line.strip() for line in raw.strip().split('\n') if line.strip()]
    return queries[:5]

In [ ]:
# Gemini-supervised query generation pipeline for Qwen finetuning data creation.
# For each train row: provide case query + gold law reference texts to Gemini, generate 5 search queries,
# evaluate each query with top-k=100 retrieval, keep top-3 by gold-hit count, print results, and save JSON dataset.

import json
import time
from pathlib import Path
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# -------------------------------
# 1) CONFIG
# -------------------------------
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
TOP_K_EVAL = 100
MAX_TRAIN_ROWS = 1200   # set int (e.g., 50) for quick runs
MAX_REF_LAWS_PER_SAMPLE = 100
MAX_CHARS_PER_LAW = 2000
REQUEST_SLEEP_SECONDS = 0.0

OUTPUT_JSON_PATH = Path("../data/qwen_querygen_finetune_top3.json")
OUTPUT_REPORT_PATH = Path("../data/qwen_querygen_generation_report.json")

# -------------------------------
# 2) HELPERS
# -------------------------------
def _parse_gold_citations(train_df: pd.DataFrame, idx: int) -> list[str]:
    raw = str(train_df.iloc[idx]["gold_citations"])
    return [c.strip() for c in raw.split(";") if c.strip()]


def _build_law_lookup(laws_df: pd.DataFrame) -> dict[str, str]:
    lookup = {}
    for _, row in laws_df.iterrows():
        cit = str(row.get("citation", "")).strip()
        txt = str(row.get("text", "")).strip()
        if cit and txt and cit not in lookup:
            lookup[cit] = txt
    return lookup


def _build_reference_laws_block(gold_citations: list[str], law_lookup: dict[str, str]) -> str:
    chunks = []
    for i, cit in enumerate(gold_citations[:MAX_REF_LAWS_PER_SAMPLE], start=1):
        law_text = law_lookup.get(cit, "")
        if not law_text:
            continue
        clipped = law_text[:MAX_CHARS_PER_LAW]
        chunks.append(f"[{i}] {cit}\n{clipped}")
    return "\n\n".join(chunks)


def _build_query_generation_prompt(user_query: str, reference_laws_block: str) -> str:
    return f"""TASK: Generate exactly 5 high-recall German legal retrieval queries.

GOAL: Queries should maximize retrieval of relevant citations from a Swiss-law vector index (top-k=100).

INPUT CASE QUERY:
{user_query}

REFERENCE LAWS (gold context):
{reference_laws_block if reference_laws_block else 'No reference law text available.'}

REQUIREMENTS:
1) Output exactly 5 queries, one per line, no numbering.
2) Use concise statutory/legal wording (keywords or short phrases).
3) Cover different legal aspects (scope, conditions, exceptions, consequences, procedure).
4) Stay semantically close to the reference laws and case query.
5) Do not output explanations or extra text.
6) Prefer terms likely appearing in statutory text to improve retrieval recall.
"""


def _generate_five_queries(llm: ChatGoogleGenerativeAI, user_query: str, reference_laws_block: str) -> list[str]:
    # prompt = _build_query_generation_prompt(user_query, reference_laws_block)
    # messages = [
    #     SystemMessage(
    #         content=
    #         "You are an expert Swiss-law retrieval query generator. Return only the 5 query lines."
    #     ),
    #     HumanMessage(content=prompt),
    # ]
    # llm.api_messages = messages  # for debugging/logging
    # response = llm.invoke(messages)
    # raw = response.content if isinstance(response.content, str) else response.content[0]["text"]
    # print(f"LLM raw response:\n{raw}\n")
    # lines = [line.strip() for line in raw.split("\n") if line.strip()]

    # # Keep first 5 non-empty lines; if fewer, pad by repeating last valid query to preserve shape.
    # if not lines:
    #     return []
    # lines = lines[:5]
    # while len(lines) < 5:
    #     lines.append(lines[-1])

    # temporary
    lines = 
    return lines


def _evaluate_query_gold_hits(query_text: str, gold_citations: list[str], top_k: int = 100) -> dict:
    docs = vectorstore.similarity_search(query_text, k=top_k)
    retrieved = [str(d.metadata.get("citation")) for d in docs if d.metadata.get("citation")]
    retrieved_unique = list(dict.fromkeys(retrieved))

    found, missing = compare_citations(retrieved_unique, gold_citations)
    return {
        "query": query_text,
        "retrieved_docs": len(docs),
        "retrieved_unique": len(set(retrieved_unique)),
        "gold_hits": len(found),
        "gold_found": found,
        "gold_missing": missing,
    }


# -------------------------------
# 3) RUN PIPELINE
# -------------------------------
if "train_csv" not in globals() or "laws_de" not in globals() or "vectorstore" not in globals():
    raise ValueError("train_csv, laws_de, and vectorstore must be loaded before running this cell.")

law_lookup = _build_law_lookup(laws_de)
# llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL_NAME, temperature=0.1)
llm = ModalQwenChat(
    api_url="https://keshavsharma25--completion.modal.run",
    temperature=0.7,
    max_new_tokens=512,
    top_p=0.95,
    max_retries=1,
    http_timeout=600,
    system_prompt=""
)

row_indices = list(range(len(train_csv)))
if isinstance(MAX_TRAIN_ROWS, int) and MAX_TRAIN_ROWS > 0:
    row_indices = row_indices[:MAX_TRAIN_ROWS]

finetune_rows = []
generation_report = []

print("\n" + "=" * 110)
print("GEMINI QUERY-GEN DATA PIPELINE (TOP-3 QUERIES BY GOLD-HITS @ TOP-K=100)")
print("=" * 110)
print(f"Model              : {GEMINI_MODEL_NAME}")
print(f"Train rows         : {len(row_indices)}")
print(f"Evaluation top-k   : {TOP_K_EVAL}")
print(f"Output JSON        : {OUTPUT_JSON_PATH}")

for r_i, idx in enumerate(row_indices, start=1):
    user_query = str(train_csv.iloc[idx]["query"]).strip()
    gold_citations = _parse_gold_citations(train_csv, idx)

    if not user_query or not gold_citations:
        print(f"[{r_i}/{len(row_indices)}] row={idx} skipped (empty query or no gold citations)")
        continue

    reference_block = _build_reference_laws_block(gold_citations, law_lookup)

    try:
        generated_queries = _generate_five_queries(llm, user_query, reference_block)
    except Exception as e:
        print(f"[{r_i}/{len(row_indices)}] row={idx} generation failed: {e}")
        continue

    if not generated_queries:
        print(f"[{r_i}/{len(row_indices)}] row={idx} generation returned no valid query lines")
        continue

    per_query_eval = [
        _evaluate_query_gold_hits(q, gold_citations, top_k=TOP_K_EVAL)
        for q in generated_queries
    ]

    top3 = sorted(
        per_query_eval,
        key=lambda x: (x["gold_hits"], x["retrieved_unique"]),
        reverse=True,
    )[:3]

    union_top3_gold = []
    seen = set()
    for row in top3:
        for g in row["gold_found"]:
            gl = g.lower()
            if gl not in seen:
                seen.add(gl)
                union_top3_gold.append(g)

    print("\n" + "-" * 110)
    print(f"[{r_i}/{len(row_indices)}] train_idx={idx}")
    print(f"Case query: {user_query}")
    print("Generated 5 queries:")
    for qn, q in enumerate(generated_queries, start=1):
        print(f"  Q{qn}: {q}")

    print("Top-3 by gold hits:")
    for rank, row in enumerate(top3, start=1):
        print(
            f"  {rank}. hits={row['gold_hits']}/{len(gold_citations)} | "
            f"retrieved_docs={row['retrieved_docs']} | retrieved_unique={row['retrieved_unique']}"
        )
        print(f"     query: {row['query']}")
        print(f"     gold_found: {row['gold_found'] if row['gold_found'] else 'NONE'}")

    print(f"Top-3 union gold coverage: {len(union_top3_gold)}/{len(gold_citations)}")

    generation_report.append(
        {
            "train_idx": int(idx),
            "case_query": user_query,
            "gold_citations": gold_citations,
            "generated_queries": generated_queries,
            "top3": top3,
            "top3_union_gold_found": union_top3_gold,
            "top3_union_gold_recall": len(union_top3_gold) / len(gold_citations) if gold_citations else 0.0,
        }
    )

    # Build finetuning rows from top-3 best queries.
    for rank, row in enumerate(top3, start=1):
        finetune_rows.append(
            {
                "id": f"train-{idx}-top{rank}",
                "train_idx": int(idx),
                "input": {
                    "case_query": user_query,
                    "reference_laws": reference_block,
                    "instruction": "Generate one German legal retrieval query maximizing citation recall.",
                },
                "output": {
                    "search_query": row["query"],
                },
                "meta": {
                    "rank_within_sample": rank,
                    "gold_hits_at_top100": int(row["gold_hits"]),
                    "gold_total": int(len(gold_citations)),
                    "gold_found": row["gold_found"],
                    "retrieved_unique": int(row["retrieved_unique"]),
                    "model": GEMINI_MODEL_NAME,
                },
            }
        )

    if REQUEST_SLEEP_SECONDS > 0:
        time.sleep(REQUEST_SLEEP_SECONDS)

# -------------------------------
# 4) SAVE ARTIFACTS
# -------------------------------
OUTPUT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(finetune_rows, f, ensure_ascii=False, indent=2)

with open(OUTPUT_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(generation_report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 110)
print("PIPELINE COMPLETE")
print("=" * 110)
print(f"Saved finetune rows : {len(finetune_rows)} -> {OUTPUT_JSON_PATH}")
print(f"Saved full report   : {len(generation_report)} -> {OUTPUT_REPORT_PATH}")
if generation_report:
    mean_top3_recall = sum(x["top3_union_gold_recall"] for x in generation_report) / len(generation_report)
    print(f"Mean top-3 union recall: {mean_top3_recall:.2%}")
else:
    print("No rows generated. Check model access / input data quality.")


GEMINI QUERY-GEN DATA PIPELINE (TOP-3 QUERIES BY GOLD-HITS @ TOP-K=100)
Model              : gemini-3-flash-preview
Train rows         : 1139
Evaluation top-k   : 100
Output JSON        : ../data/qwen_querygen_finetune_top3.json
api messages: [{'role': 'system', 'content': 'You are an expert Swiss-law retrieval query generator. Return only the 5 query lines.'}, {'role': 'user', 'content': "TASK: Generate exactly 5 high-recall German legal retrieval queries.\n\nGOAL: Queries should maximize retrieval of relevant citations from a Swiss-law vector index (top-k=100).\n\nINPUT CASE QUERY:\nDie A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunternehmen. Das Unternehmen verarbeitet vorwiegend Metallschrott. Dieser wird im Aussenbereich des Betriebsgeländes zwischengelagert und dort von einem fest instal lierten Metallschredder für die Weiterverarbeitung zerkleinert. Im Jahr 2018 wurde der 1983 bewilligte Metallschredder umfassend modernisie

KeyboardInterrupt: 

### checking if there is aany relation within citations

In [92]:
train_csv["gold_citations"][1].split(';')

['Art. 975 ZGB',
 'Art. 974 Abs. 2 ZGB',
 'Art. 973 ZGB',
 'Art. 661 ZGB',
 'Art. 956a ZGB',
 'Art. 956a Abs. 3 ZGB',
 'Art. 955 Abs. 1 ZGB',
 'Art. 976 ZGB',
 'Art. 976a ZGB',
 'Art. 60 OR',
 'Art. 955 Abs. 2 ZGB']

In [91]:
list(laws_de["citation"])

['Art. 1 112',
 'Art. 2 112',
 'Art. 3 Abs. 1 112',
 'Art. 3 Abs. 2 112',
 'Art. 4 Abs. 1 112',
 'Art. 4 Abs. 2 112',
 'Art. 4 Abs. 3 112',
 'Art. 5 Abs. 1 112',
 'Art. 5 Abs. 2 112',
 'Art. 8 112',
 'Art. 9 Abs. 1 112',
 'Art. 9 Abs. 2 112',
 'Art. 1 Abs. 1 116',
 'Art. 1 Abs. 2 116',
 'Art. 1 Abs. 3 116',
 'Art. 2 Abs. 1 116',
 'Art. 2 Abs. 2 116',
 'Art. 3 116',
 'Art. 4 116',
 'Art. 1 Abs. 1 122.1',
 'Art. 1 Abs. 2 122.1',
 'Art. 1 Abs. 3 122.1',
 'Art. 2 122.1',
 'Art. 3 122.1',
 'Art. 4 Abs. 1 122.1',
 'Art. 4 Abs. 2 122.1',
 'Art. 4 Abs. 3 122.1',
 'Art. 1 Abs. 1 131.211',
 'Art. 1 Abs. 2 131.211',
 'Art. 1 Abs. 3 131.211',
 'Art. 1 Abs. 4 131.211',
 'Art. 2 Abs. 1 131.211',
 'Art. 2 Abs. 2 131.211',
 'Art. 2 Abs. 3 131.211',
 'Art. 3 Abs. 1 131.211',
 'Art. 3 Abs. 2 131.211',
 'Art. 4 131.211',
 'Art. 5 Abs. 1 131.211',
 'Art. 5 Abs. 2 131.211',
 'Art. 5 Abs. 3 131.211',
 'Art. 6 Abs. 1 131.211',
 'Art. 6 Abs. 2 131.211',
 'Art. 7 131.211',
 'Art. 8 131.211',
 'Art. 9 131.211',

In [123]:
# N-gram retrieval experiment on one training query (2,3,4,5,6-grams).
# Steps: remove German stopwords -> build n-grams -> retrieve top-k=100 per n-gram -> report citation counts,
# citation names, and gold-hit counts per n-gram.

import re
from collections import Counter
from IPython.display import display

# -------------------------------
# 1) CONFIG
# -------------------------------
train_idx_for_ngram_experiment = 1
ngram_sizes = [2, 3, 4, 5, 6]
top_k_per_ngram = 100
min_token_len = 2
max_ngrams_per_n = None  # set an int to cap runtime per n (e.g., 40)

# -------------------------------
# 2) INPUT CHECKS
# -------------------------------
if "train_csv" not in globals() or "vectorstore" not in globals():
    raise ValueError("train_csv and vectorstore must be loaded before running this cell.")

if train_idx_for_ngram_experiment < 0 or train_idx_for_ngram_experiment >= len(train_csv):
    raise IndexError(
        f"train_idx_for_ngram_experiment={train_idx_for_ngram_experiment} out of range [0, {len(train_csv)-1}]"
    )

if "gold_citations" not in train_csv.columns:
    raise ValueError("train_csv must contain a 'gold_citations' column for this report.")

source_query_text = str(train_csv.iloc[train_idx_for_ngram_experiment]["query"]).strip()
if not source_query_text:
    raise ValueError("Selected training row has an empty query text.")

raw_gold = str(train_csv.iloc[train_idx_for_ngram_experiment]["gold_citations"])
gold_citations = [c.strip() for c in raw_gold.split(";") if c.strip()]
gold_lower = {g.lower() for g in gold_citations}

# -------------------------------
# 3) STOPWORDS + TOKENIZATION
# -------------------------------
if "GERMAN_STOPWORDS" in globals() and isinstance(GERMAN_STOPWORDS, set):
    german_stopwords = GERMAN_STOPWORDS
else:
    german_stopwords = {
        "aber", "als", "am", "an", "auch", "auf", "aus", "bei", "bis", "da", "das", "dass", "dem", "den",
        "der", "des", "die", "doch", "du", "durch", "ein", "eine", "einer", "eines", "er", "es", "für",
        "hat", "hier", "ich", "ihr", "im", "in", "ist", "ja", "mit", "nach", "nicht", "oder", "sein",
        "sich", "sie", "sind", "so", "und", "vom", "von", "vor", "war", "was", "wenn", "wer", "wie", "wir",
        "wird", "wo", "zu", "zum", "zur", "über"
    }

if "EN_STOPWORDS" in globals() and isinstance(EN_STOPWORDS, set):
    english_stopwords = EN_STOPWORDS
else:
    english_stopwords = {
        "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or", "that",
        "the", "to", "was", "were", "with"
    }

all_stopwords = german_stopwords.union(english_stopwords)

clean_text = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", " ", source_query_text.lower())
tokens = [t for t in clean_text.split() if t and len(t) >= min_token_len and t not in all_stopwords]

if not tokens:
    raise ValueError("No tokens left after stopword removal. Try lowering stopword filtering.")

# -------------------------------
# 4) BUILD N-GRAMS
# -------------------------------
def _build_ngrams(tokens_list: list[str], n: int) -> list[str]:
    if len(tokens_list) < n:
        return []
    grams = [" ".join(tokens_list[i:i+n]) for i in range(len(tokens_list) - n + 1)]
    # De-duplicate while preserving order.
    grams = list(dict.fromkeys(grams))
    if isinstance(max_ngrams_per_n, int) and max_ngrams_per_n > 0:
        grams = grams[:max_ngrams_per_n]
    return grams

ngrams_by_n = {n: _build_ngrams(tokens, n) for n in ngram_sizes}

# -------------------------------
# 5) RETRIEVE + EVALUATE
# -------------------------------
report_rows = []

print("\n" + "=" * 110)
print("N-GRAM RETRIEVAL EXPERIMENT (TOP-K=100) WITH GOLD HIT REPORT")
print("=" * 110)
print(f"Train index                : {train_idx_for_ngram_experiment}")
print(f"Original query length      : {len(source_query_text)} chars")
print(f"Token count after cleanup  : {len(tokens)}")
print(f"N values                   : {ngram_sizes}")
print(f"top_k_per_ngram            : {top_k_per_ngram}")
print(f"Gold citation count        : {len(gold_citations)}")

for n in ngram_sizes:
    ngrams = ngrams_by_n.get(n, [])
    print(f"\n--- n={n} | ngrams_to_search={len(ngrams)} ---")

    for gram in ngrams:
        docs = vectorstore.similarity_search(gram, k=top_k_per_ngram)
        citations = [
            str(d.metadata.get("citation")).strip()
            for d in docs
            if d.metadata.get("citation") and str(d.metadata.get("citation")).strip()
        ]

        unique_citations = list(dict.fromkeys(citations))
        unique_lower = {c.lower() for c in unique_citations}
        citation_freq = Counter(citations)

        gold_citation_names_found = [g for g in gold_citations if g.lower() in unique_lower]
        gold_hits_unique = len(gold_citation_names_found)
        gold_hits_in_topk_docs = sum(1 for c in citations if c.lower() in gold_lower)

        report_rows.append(
            {
                "n": n,
                "ngram": gram,
                "retrieved_docs": len(docs),
                "unique_citation_count": len(unique_citations),
                "citation_names": unique_citations,
                "top10_citations_by_freq": citation_freq.most_common(10),
                "gold_hits_in_topk_docs": gold_hits_in_topk_docs,
                "gold_hits_in_unique_citations": gold_hits_unique,
                "gold_hit_rate_topk_docs": round(gold_hits_in_topk_docs / top_k_per_ngram, 4),
                "gold_recall_vs_row_gold": round(gold_hits_unique / len(gold_citations), 4) if gold_citations else 0.0,
                "gold_citation_names_found": gold_citation_names_found,
            }
        )

if not report_rows:
    raise ValueError("No n-grams were generated/evaluated. Try a smaller n or another training row.")

ngram_report_df = pd.DataFrame(report_rows).sort_values(
    ["gold_hits_in_topk_docs", "gold_hits_in_unique_citations", "unique_citation_count"],
    ascending=False,
).reset_index(drop=True)

n_summary_df = (
    ngram_report_df.groupby("n", as_index=False)
    .agg(
        ngram_count=("ngram", "count"),
        avg_unique_citation_count=("unique_citation_count", "mean"),
        max_unique_citation_count=("unique_citation_count", "max"),
        avg_gold_hits_topk_docs=("gold_hits_in_topk_docs", "mean"),
        max_gold_hits_topk_docs=("gold_hits_in_topk_docs", "max"),
        avg_gold_hits_unique=("gold_hits_in_unique_citations", "mean"),
        max_gold_hits_unique=("gold_hits_in_unique_citations", "max"),
        avg_gold_recall=("gold_recall_vs_row_gold", "mean"),
        max_gold_recall=("gold_recall_vs_row_gold", "max"),
    )
    .sort_values("n")
    .reset_index(drop=True)
)

best_per_n_df = (
    ngram_report_df.sort_values(
        ["n", "gold_hits_in_topk_docs", "gold_hits_in_unique_citations", "unique_citation_count"],
        ascending=[True, False, False, False],
    )
    .groupby("n", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("\n" + "-" * 110)
print("Summary by n (including gold-hit metrics):")
display(n_summary_df)

print("\nBest n-gram per n (includes found gold citation names):")
display(
    best_per_n_df[
        [
            "n",
            "ngram",
            "gold_hits_in_topk_docs",
            "gold_hits_in_unique_citations",
            "gold_recall_vs_row_gold",
            "gold_citation_names_found",
            "unique_citation_count",
        ]
    ]
)

print("\nTop overall n-grams by gold hits (first 25):")
display(
    ngram_report_df[
        [
            "n",
            "ngram",
            "gold_hits_in_topk_docs",
            "gold_hits_in_unique_citations",
            "gold_recall_vs_row_gold",
            "gold_citation_names_found",
            "unique_citation_count",
        ]
    ].head(25)
)

print("\nDone. DataFrames available: ngram_report_df, n_summary_df, best_per_n_df")


N-GRAM RETRIEVAL EXPERIMENT (TOP-K=100) WITH GOLD HIT REPORT
Train index                : 1
Original query length      : 1376 chars
Token count after cleanup  : 120
N values                   : [2, 3, 4, 5, 6]
top_k_per_ngram            : 100
Gold citation count        : 11

--- n=2 | ngrams_to_search=115 ---

--- n=3 | ngrams_to_search=116 ---

--- n=4 | ngrams_to_search=117 ---

--- n=5 | ngrams_to_search=116 ---

--- n=6 | ngrams_to_search=115 ---

--------------------------------------------------------------------------------------------------------------
Summary by n (including gold-hit metrics):


,n,ngram_count,avg_unique_citation_count,max_unique_citation_count,avg_gold_hits_topk_docs,max_gold_hits_topk_docs,avg_gold_hits_unique,max_gold_hits_unique,avg_gold_recall,max_gold_recall
0,2,115,100.0,100,0.034783,1,0.034783,1,0.003162,0.0909
1,3,116,100.0,100,0.086207,1,0.086207,1,0.007836,0.0909
2,4,117,100.0,100,0.102564,1,0.102564,1,0.009323,0.0909
3,5,116,100.0,100,0.077586,1,0.077586,1,0.007053,0.0909
4,6,115,100.0,100,0.104348,1,0.104348,1,0.009485,0.0909



Best n-gram per n (includes found gold citation names):


,n,ngram,gold_hits_in_topk_docs,gold_hits_in_unique_citations,gold_recall_vs_row_gold,gold_citation_names_found,unique_citation_count
0,2,anschliessen grunddienstbarkeit,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
1,3,kann nun grundstück,1,1,0.0909,[Art. 661 ZGB],100
2,4,kann nun grundstück neu,1,1,0.0909,[Art. 661 ZGB],100
3,5,ausser betrieb nimmt grundstück jetzt,1,1,0.0909,[Art. 976 ZGB],100
4,6,kann nun grundstück neu direkt soeben,1,1,0.0909,[Art. 661 ZGB],100



Top overall n-grams by gold hits (first 25):


,n,ngram,gold_hits_in_topk_docs,gold_hits_in_unique_citations,gold_recall_vs_row_gold,gold_citation_names_found,unique_citation_count
0,2,anschliessen grunddienstbarkeit,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
1,2,grundbuchverwalter grundbuchkreises,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
2,2,entlastung grundbuchs,1,1,0.0909,[Art. 956a Abs. 3 ZGB],100
3,2,wegen löschen,1,1,0.0909,[Art. 976 ZGB],100
4,3,kann nun grundstück,1,1,0.0909,[Art. 661 ZGB],100
5,3,anschliessen grunddienstbarkeit erfasste,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
6,3,anschliesst georg grundbuchverwalter,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
7,3,georg grundbuchverwalter grundbuchkreises,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
8,3,grundbuchverwalter grundbuchkreises grundstücke,1,1,0.0909,[Art. 955 Abs. 2 ZGB],100
9,3,grundstücke liegen wohnt,1,1,0.0909,[Art. 661 ZGB],100



Done. DataFrames available: ngram_report_df, n_summary_df, best_per_n_df


In [136]:
train_csv.iloc[1]['query']

'Die Alpha Consulting AG kann nun ihr Grundstück neu direkt an die soeben erstellte Kana lisationsleitung in der angrenzenden neuen Quartierstrasse anschliessen, so dass sie die von der Grunddienstbarkeit erfasste Abwasserleitung (vgl. Sachverhalt Teil 1) ausser Betrieb nimmt und ihr Grundstück jetzt an die Leitung in der Quartierstrasse anschliesst. Georg (G), der Grundbuchverwalter des Grundbuchkreises, in dem die Grundstücke 1 und 2 liegen, wohnt ganz in der Nähe und beobachtet auf seinem täglichen Weg zur Arbeit die Installation der neuen Kanalisationsleitung in der Quartierstrasse. Einige Tage später liest er zudem in der Zeitung, dass sämtliche Liegenschaften, die an die Quartierstrasse grenzen, neu an die in der Quartierstrasse befindliche Kanalisationsleitung angeschlossen worden seien. Daher fasst Georg den Entschluss, zur Entlastung des Grundbuchs die bisher eingetra gene Leitungsdienstbarkeit von Amtes wegen zu löschen, und setzt sein Vorhaben sofort in die Tat um. \nSollte 

In [135]:
train_csv.iloc[1]['gold_citations'].split(';')

['Art. 975 ZGB',
 'Art. 974 Abs. 2 ZGB',
 'Art. 973 ZGB',
 'Art. 661 ZGB',
 'Art. 956a ZGB',
 'Art. 956a Abs. 3 ZGB',
 'Art. 955 Abs. 1 ZGB',
 'Art. 976 ZGB',
 'Art. 976a ZGB',
 'Art. 60 OR',
 'Art. 955 Abs. 2 ZGB']

In [159]:
print(laws_de[laws_de['citation'] == 'Art. 974 Abs. 2 ZGB']['text'].values[0])

2 Ungerechtfertigt ist der Eintrag, der ohne Rechtsgrund oder aus einem unverbindlichen Rechtsgeschäft erfolgt ist.


In [182]:
laws_de[laws_de['citation'] == "Art. 655 Abs. 2 ZGB"]['text'].values[0]

'2 Grundstücke im Sinne dieses Gesetzes sind:1. die Liegenschaften;\n2. die in das Grundbuch aufgenommenen selbständigen und dauernden Rechte;\n3. die Bergwerke;\n4. die Miteigentumsanteile an Grundstücken.'

In [183]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [161]:
for i in train_csv.iloc[1]['gold_citations'].split(';'):
    print(i.strip())
    try:
        cit_text = laws_de[laws_de['citation'] == i.strip()]['text'].values[0]
        print(cit_text)
    except IndexError:
        print(f"Warning: Citation '{i.strip()}' not found in laws_de.")
    print('---\n')

Art. 975 ZGB
---

Art. 974 Abs. 2 ZGB
2 Ungerechtfertigt ist der Eintrag, der ohne Rechtsgrund oder aus einem unverbindlichen Rechtsgeschäft erfolgt ist.
---

Art. 973 ZGB
---

Art. 661 ZGB
Ist jemand ungerechtfertigt im Grundbuch als Eigentümer eingetragen, so kann sein Eigentum, nachdem er das Grundstück in gutem Glauben zehn Jahre lang ununterbrochen und unangefochten besessen hat, nicht mehr angefochten werden.
---

Art. 956a ZGB
---

Art. 956a Abs. 3 ZGB
3 Gegen eine im Hauptbuch vollzogene Eintragung, Änderung oder Löschung von dinglichen Rechten oder Vormerkungen kann keine Beschwerde mehr geführt werden.
---

Art. 955 Abs. 1 ZGB
1 Die Kantone sind für allen Schaden verantwortlich, der aus der Führung des Grundbuches entsteht.
---

Art. 976 ZGB
Das Grundbuchamt kann einen Eintrag von Amtes wegen löschen, wenn dieser:1. befristet ist und infolge Ablaufs der Frist seine rechtliche Bedeutung verloren hat;
2. ein unübertragbares oder unvererbliches Recht einer verstorbenen Person be

In [184]:
train_csv

,query_id,query,gold_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB
3,train_0004,Die Linzer Stadtbahn AG ('LiSA') ist die priva...,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....
4,train_0005,Die Stadt Winterthur beschloss am 10. Februar ...,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...
...,...,...,...
1134,train_1135,Der Türke Deniz kam 2003 als Vierjähriger mit ...,Art. 12 BüG;Art. 11 BüG;Art. 9 Abs. 1 BüG;Art....
1135,train_1136,P ist mexikanischer Staatsangehöriger und heir...,Art. 13 Abs. 1 BV
1136,train_1137,"Sheyla ist reich und sammelt schnelle Autos, a...",Art. 7 Abs. 1 OBG;Art. 7 OBV;Art. 7 OBG;Art. 9...
1137,train_1138,C arbeitet als Sportartikelverkäuferin bei der...,Art. 54 Abs. 1 ArG


## searching for n grams from the laws text (not the query)

In [203]:
train_idx_for_ngram_experiment = 1
# training data point
query = train_csv.iloc[train_idx_for_ngram_experiment]["query"]
citations = train_csv.iloc[train_idx_for_ngram_experiment]["gold_citations"].split(';')
print(f"Query: {query}")
print(f"Citations: {citations}")


def _contains_token_safely(token: str, text: str) -> bool:
    """
    Match token with non-alphanumeric boundaries so '60' does not match inside '160'.
    """
    token = token.strip()
    if not token:
        return True
    pattern = rf"(?<![0-9A-Za-z]){re.escape(token)}(?![0-9A-Za-z])"
    return re.search(pattern, text, flags=re.IGNORECASE) is not None


def _extract_article_number(citation_text: str) -> str | None:
    """Extract article number token after 'Art.' (e.g., '60', '955', '956a')."""
    m = re.search(r"\bArt\.\s*([0-9]+[a-zA-Z]*)\b", str(citation_text), flags=re.IGNORECASE)
    return m.group(1).lower() if m else None


def return_matching_citation_from_laws_de(citation: str) -> str:
    citation_clean = citation.strip()

    # 1) Try case-insensitive exact citation match first.
    exact_mask = laws_de["citation"].astype(str).str.strip().str.casefold() == citation_clean.casefold()
    exact_rows = laws_de[exact_mask]
    if not exact_rows.empty:
        return str(exact_rows.iloc[0]["text"])

    # 2) Fallback: token-safe partial matching with strict article-number equality.
    words = [w for w in citation_clean.split(" ") if w.strip()]
    target_article = _extract_article_number(citation_clean)

    # Remaining tokens to validate with boundary-safe matching.
    remaining_tokens = [
        w for w in words
        if w.lower() not in {"art.", "art"} and (target_article is None or w.lower() != target_article)
    ]

    final_text_chunks = []
    for _, row in laws_de.iterrows():
        law_citation = str(row["citation"]).strip()

        # Enforce exact article number after 'Art.' when present in input citation.
        if target_article is not None:
            law_article = _extract_article_number(law_citation)
            if law_article != target_article:
                continue

        if all(_contains_token_safely(word, law_citation) for word in remaining_tokens):
            print(f"Found partial match in laws_de: '{law_citation}' for input citation '{citation_clean}'")
            final_text_chunks.append(str(row["text"]))

    if final_text_chunks:
        return "\n".join(final_text_chunks)

    return f"Warning: Citation '{citation_clean}' not found in laws_de."

Query: Die Alpha Consulting AG kann nun ihr Grundstück neu direkt an die soeben erstellte Kana lisationsleitung in der angrenzenden neuen Quartierstrasse anschliessen, so dass sie die von der Grunddienstbarkeit erfasste Abwasserleitung (vgl. Sachverhalt Teil 1) ausser Betrieb nimmt und ihr Grundstück jetzt an die Leitung in der Quartierstrasse anschliesst. Georg (G), der Grundbuchverwalter des Grundbuchkreises, in dem die Grundstücke 1 und 2 liegen, wohnt ganz in der Nähe und beobachtet auf seinem täglichen Weg zur Arbeit die Installation der neuen Kanalisationsleitung in der Quartierstrasse. Einige Tage später liest er zudem in der Zeitung, dass sämtliche Liegenschaften, die an die Quartierstrasse grenzen, neu an die in der Quartierstrasse befindliche Kanalisationsleitung angeschlossen worden seien. Daher fasst Georg den Entschluss, zur Entlastung des Grundbuchs die bisher eingetra gene Leitungsdienstbarkeit von Amtes wegen zu löschen, und setzt sein Vorhaben sofort in die Tat um. 
So

In [208]:
# Detailed n-gram retrieval experiment from GOLD LAW TEXT (not query).
# Uses the same matching logic as return_matching_citation_from_laws_de for gold-hit evaluation.

from collections import Counter
from IPython.display import display

# -------------------------------
# 1) CONFIG
# -------------------------------
ngram_sizes = [7, 8, 9, 10]
top_k_per_ngram = 50
min_token_len = 2
max_ngrams_per_n = 80  # set None for exhaustive, but runtime can be very large

# -------------------------------
# 2) BUILD GOLD MATCH SET USING SAME LOGIC
# -------------------------------
gold_base_citations = [c.strip() for c in citations if c.strip()]

def _matching_laws_rows_for_citation(citation: str) -> pd.DataFrame:
    """Return matching rows from laws_de using the same matching logic as return_matching_citation_from_laws_de."""
    citation_clean = citation.strip()

    exact_mask = laws_de["citation"].astype(str).str.strip().str.casefold() == citation_clean.casefold()
    exact_rows = laws_de.loc[exact_mask, ["citation", "text"]]
    if not exact_rows.empty:
        return exact_rows

    words = [w for w in citation_clean.split(" ") if w.strip()]
    target_article = _extract_article_number(citation_clean)
    remaining_tokens = [
        w for w in words
        if w.lower() not in {"art.", "art"} and (target_article is None or w.lower() != target_article)
    ]

    matched_indices = []
    for idx_row, row in laws_de.iterrows():
        law_citation = str(row["citation"]).strip()

        if target_article is not None:
            law_article = _extract_article_number(law_citation)
            if law_article != target_article:
                continue

        if all(_contains_token_safely(word, law_citation) for word in remaining_tokens):
            matched_indices.append(idx_row)

    if matched_indices:
        return laws_de.loc[matched_indices, ["citation", "text"]]

    return laws_de.iloc[0:0][["citation", "text"]]


gold_variant_to_base = {}        # variant citation lower -> set(base gold citation)
gold_variant_text_lookup = {}     # variant citation lower -> law text
source_law_text_chunks = []
unmatched_gold_bases = []

for base_cit in gold_base_citations:
    matched_rows = _matching_laws_rows_for_citation(base_cit)
    if matched_rows.empty:
        unmatched_gold_bases.append(base_cit)
        continue

    for _, r in matched_rows.iterrows():
        variant_cit = str(r["citation"]).strip()
        variant_text = str(r["text"])
        key = variant_cit.casefold()

        gold_variant_to_base.setdefault(key, set()).add(base_cit)
        if key not in gold_variant_text_lookup:
            gold_variant_text_lookup[key] = variant_text
        source_law_text_chunks.append(variant_text)

gold_variant_citations = sorted({k for k in gold_variant_to_base.keys()})

print("\n" + "=" * 120)
print("DETAILED N-GRAM EXPERIMENT FROM GOLD LAW TEXT")
print("=" * 120)
print(f"Train index                  : {train_idx_for_ngram_experiment}")
print(f"Gold base citations          : {len(gold_base_citations)}")
print(f"Gold variant citations set   : {len(gold_variant_citations)}")
print(f"Unmatched base citations     : {len(unmatched_gold_bases)}")
if unmatched_gold_bases:
    print("  Unmatched:", unmatched_gold_bases)

if not source_law_text_chunks:
    raise ValueError("No source law text found from gold citations. Cannot build n-grams.")

# -------------------------------
# 3) TOKENIZE GOLD LAW TEXT AND BUILD N-GRAMS
# -------------------------------
if "GERMAN_STOPWORDS" in globals() and isinstance(GERMAN_STOPWORDS, set):
    german_stopwords = GERMAN_STOPWORDS
else:
    german_stopwords = {
        "aber", "als", "am", "an", "auch", "auf", "aus", "bei", "bis", "da", "das", "dass", "dem", "den",
        "der", "des", "die", "doch", "du", "durch", "ein", "eine", "einer", "eines", "er", "es", "für",
        "hat", "hier", "ich", "ihr", "im", "in", "ist", "ja", "mit", "nach", "nicht", "oder", "sein",
        "sich", "sie", "sind", "so", "und", "vom", "von", "vor", "war", "was", "wenn", "wer", "wie", "wir",
        "wird", "wo", "zu", "zum", "zur", "über"
    }

source_law_text = "\n\n".join(source_law_text_chunks)
clean_text = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", " ", source_law_text.lower())
tokens = [t for t in clean_text.split() if t and len(t) >= min_token_len and t not in german_stopwords]

if not tokens:
    raise ValueError("No tokens left after stopword cleanup of source law text.")

def _build_ranked_ngrams(tokens_list: list[str], n: int, max_count: int | None = None) -> list[str]:
    if len(tokens_list) < n:
        return []
    grams = [" ".join(tokens_list[i:i+n]) for i in range(len(tokens_list) - n + 1)]
    ranked = [g for g, _ in Counter(grams).most_common()]
    if isinstance(max_count, int) and max_count > 0:
        return ranked[:max_count]
    return ranked

ngrams_by_n = {n: _build_ranked_ngrams(tokens, n, max_ngrams_per_n) for n in ngram_sizes}
print(f"Source law text chars       : {len(source_law_text)}")
print(f"Tokens after cleanup        : {len(tokens)}")
print(f"N-gram sizes                : {ngram_sizes}")
print(f"top_k_per_ngram             : {top_k_per_ngram}")
print(f"max_ngrams_per_n            : {max_ngrams_per_n}")

# -------------------------------
# 4) RETRIEVE + GOLD-HIT SCORING
# -------------------------------
detailed_rows = []

for n in ngram_sizes:
    ngrams = ngrams_by_n.get(n, [])
    print(f"\n--- Running n={n} with {len(ngrams)} n-grams ---")

    for gram in ngrams:
        docs = vectorstore.similarity_search(gram, k=top_k_per_ngram)
        retrieved_citations = [
            str(d.metadata.get("citation")).strip()
            for d in docs
            if d.metadata.get("citation") and str(d.metadata.get("citation")).strip()
        ]

        unique_retrieved = list(dict.fromkeys(retrieved_citations))
        freq_counter = Counter(retrieved_citations)

        gold_hit_details = []
        for rc in unique_retrieved:
            key = rc.casefold()
            if key in gold_variant_to_base:
                gold_hit_details.append({
                    "citation": rc,
                    "matched_gold_base": sorted(gold_variant_to_base[key]),
                    "cit_text": gold_variant_text_lookup.get(key, ""),
                })

        # Count unique variant hits and base-gold coverage hits.
        gold_hit_unique_count = len(gold_hit_details)
        base_gold_hit_set = set()
        for d_hit in gold_hit_details:
            for b in d_hit["matched_gold_base"]:
                base_gold_hit_set.add(b)
        base_gold_hit_count = len(base_gold_hit_set)

        # Also count doc-level hits in top-k (duplicates allowed).
        doc_level_gold_hits = sum(1 for rc in retrieved_citations if rc.casefold() in gold_variant_to_base)

        detailed_rows.append({
            "n": n,
            "ngram": gram,
            "retrieved_docs": len(docs),
            "unique_retrieved_citations": len(unique_retrieved),
            "top10_retrieved_citations": freq_counter.most_common(10),
            "gold_hits_unique_variants": gold_hit_unique_count,
            "gold_hits_unique_bases": base_gold_hit_count,
            "gold_hits_doc_level_topk": doc_level_gold_hits,
            "gold_recall_base": round(base_gold_hit_count / len(gold_base_citations), 4) if gold_base_citations else 0.0,
            "gold_hit_names": [x["citation"] for x in gold_hit_details],
            "gold_hit_base_names": sorted(base_gold_hit_set),
            "gold_hit_details": gold_hit_details,
        })

if not detailed_rows:
    raise ValueError("No n-gram rows were evaluated.")

ngram_detailed_df = pd.DataFrame(detailed_rows).sort_values(
    ["gold_hits_unique_bases", "gold_hits_unique_variants", "gold_hits_doc_level_topk", "unique_retrieved_citations"],
    ascending=False,
).reset_index(drop=True)

ngram_summary_df = (
    ngram_detailed_df.groupby("n", as_index=False)
    .agg(
        ngram_count=("ngram", "count"),
        avg_gold_hits_bases=("gold_hits_unique_bases", "mean"),
        max_gold_hits_bases=("gold_hits_unique_bases", "max"),
        avg_gold_hits_variants=("gold_hits_unique_variants", "mean"),
        max_gold_hits_variants=("gold_hits_unique_variants", "max"),
        avg_doc_level_gold_hits=("gold_hits_doc_level_topk", "mean"),
        max_doc_level_gold_hits=("gold_hits_doc_level_topk", "max"),
        avg_gold_recall_base=("gold_recall_base", "mean"),
        max_gold_recall_base=("gold_recall_base", "max"),
    )
    .sort_values("n")
    .reset_index(drop=True)
)

best_per_n_df = (
    ngram_detailed_df.sort_values(
        ["n", "gold_hits_unique_bases", "gold_hits_unique_variants", "gold_hits_doc_level_topk"],
        ascending=[True, False, False, False],
    )
    .groupby("n", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# -------------------------------
# 5) FINAL DETAILED REPORT
# -------------------------------
print("\n" + "-" * 120)
print("SUMMARY BY N")
display(ngram_summary_df)

print("\nBEST N-GRAM PER N (with gold names + cit_text details)")
display(
    best_per_n_df[
        [
            "n",
            "ngram",
            "gold_hits_unique_bases",
            "gold_hits_unique_variants",
            "gold_hits_doc_level_topk",
            "gold_recall_base",
            "gold_hit_base_names",
            "gold_hit_names",
            "gold_hit_details",
        ]
    ]
)

print("\nTOP OVERALL N-GRAMS (first 50) with full details")
display(
    ngram_detailed_df[
        [
            "n",
            "ngram",
            "gold_hits_unique_bases",
            "gold_hits_unique_variants",
            "gold_hits_doc_level_topk",
            "gold_recall_base",
            "gold_hit_base_names",
            "gold_hit_names",
            "gold_hit_details",
            "top10_retrieved_citations",
        ]
    ].head(50)
)

print("\nDone. DataFrames available: ngram_detailed_df, ngram_summary_df, best_per_n_df")


DETAILED N-GRAM EXPERIMENT FROM GOLD LAW TEXT
Train index                  : 1
Gold base citations          : 11
Gold variant citations set   : 18
Unmatched base citations     : 0
Source law text chars       : 4340
Tokens after cleanup        : 351
N-gram sizes                : [7, 8, 9, 10]
top_k_per_ngram             : 50
max_ngrams_per_n            : 80

--- Running n=7 with 80 n-grams ---

--- Running n=8 with 80 n-grams ---

--- Running n=9 with 80 n-grams ---

--- Running n=10 with 80 n-grams ---

------------------------------------------------------------------------------------------------------------------------
SUMMARY BY N


,n,ngram_count,avg_gold_hits_bases,max_gold_hits_bases,avg_gold_hits_variants,max_gold_hits_variants,avg_doc_level_gold_hits,max_doc_level_gold_hits,avg_gold_recall_base,max_gold_recall_base
0,7,80,1.4125,5,1.4000,5,1.4000,5,0.128396,0.4545
1,8,80,1.6375,4,1.6625,4,1.6625,4,0.148849,0.3636
2,9,80,1.7875,4,1.8875,5,1.8875,5,0.162484,0.3636
3,10,80,1.8875,4,1.9875,5,1.9875,5,0.171574,0.3636



BEST N-GRAM PER N (with gold names + cit_text details)


,n,ngram,gold_hits_unique_bases,gold_hits_unique_variants,gold_hits_doc_level_topk,gold_recall_base,gold_hit_base_names,gold_hit_names,gold_hit_details
0,7,dinglichen rechte ansprüche schadenersatz unge...,5,5,5,0.4545,"[Art. 956a Abs. 3 ZGB, Art. 956a ZGB, Art. 974...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_..."
1,8,gutem glauben einen eintrag grundbuch verlasse...,4,4,4,0.3636,"[Art. 661 ZGB, Art. 973 ZGB, Art. 975 ZGB, Art...","[Art. 973 Abs. 1 ZGB, Art. 661 ZGB, Art. 976 Z...","[{'citation': 'Art. 973 Abs. 1 ZGB', 'matched_..."
2,9,grundstücken kanton bezeichneten gebieten bode...,4,5,5,0.3636,"[Art. 955 Abs. 1 ZGB, Art. 956a ZGB, Art. 973 ...","[Art. 973 Abs. 2 ZGB, Art. 956a Abs. 1 ZGB, Ar...","[{'citation': 'Art. 973 Abs. 2 ZGB', 'matched_..."
3,10,erworbenen dinglichen rechte ansprüche schaden...,4,5,5,0.3636,"[Art. 973 ZGB, Art. 974 Abs. 2 ZGB, Art. 975 Z...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_..."



TOP OVERALL N-GRAMS (first 50) with full details


,n,ngram,gold_hits_unique_bases,gold_hits_unique_variants,gold_hits_doc_level_topk,gold_recall_base,gold_hit_base_names,gold_hit_names,gold_hit_details,top10_retrieved_citations
0,7,dinglichen rechte ansprüche schadenersatz unge...,5,5,5,0.4545,"[Art. 956a Abs. 3 ZGB, Art. 956a ZGB, Art. 974...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_...","[(Art. 974 Abs. 2 ZGB, 1), (Art. 974 Abs. 1 ZG..."
1,9,grundstücken kanton bezeichneten gebieten bode...,4,5,5,0.3636,"[Art. 955 Abs. 1 ZGB, Art. 956a ZGB, Art. 973 ...","[Art. 973 Abs. 2 ZGB, Art. 956a Abs. 1 ZGB, Ar...","[{'citation': 'Art. 973 Abs. 2 ZGB', 'matched_...","[(Art. 62 Abs. 7 SVV, 1), (Art. 660b Abs. 1 ZG..."
2,10,erworbenen dinglichen rechte ansprüche schaden...,4,5,5,0.3636,"[Art. 973 ZGB, Art. 974 Abs. 2 ZGB, Art. 975 Z...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_...","[(Art. 974 Abs. 2 ZGB, 1), (Art. 975 Abs. 1 ZG..."
3,8,gutem glauben einen eintrag grundbuch verlasse...,4,4,4,0.3636,"[Art. 661 ZGB, Art. 973 ZGB, Art. 975 ZGB, Art...","[Art. 973 Abs. 1 ZGB, Art. 661 ZGB, Art. 976 Z...","[{'citation': 'Art. 973 Abs. 1 ZGB', 'matched_...","[(Art. 973 Abs. 1 ZGB, 1), (Art. 666 Abs. 1 ZG..."
4,8,kanton bezeichneten gebieten bodenverschiebung...,4,4,4,0.3636,"[Art. 955 Abs. 1 ZGB, Art. 956a ZGB, Art. 973 ...","[Art. 973 Abs. 2 ZGB, Art. 956a Abs. 1 ZGB, Ar...","[{'citation': 'Art. 973 Abs. 2 ZGB', 'matched_...","[(Art. 973 Abs. 2 ZGB, 1), (Art. 668 Abs. 3 ZG..."
5,10,dritten eintragung erworbenen dinglichen recht...,4,4,4,0.3636,"[Art. 956a Abs. 3 ZGB, Art. 956a ZGB, Art. 974...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_...","[(Art. 974 Abs. 1 ZGB, 1), (Art. 974 Abs. 3 ZG..."
6,10,rechtsgeschäft erfolgt gutem glauben einen ein...,4,4,4,0.3636,"[Art. 661 ZGB, Art. 973 ZGB, Art. 974 Abs. 2 Z...","[Art. 973 Abs. 1 ZGB, Art. 661 ZGB, Art. 975 A...","[{'citation': 'Art. 973 Abs. 1 ZGB', 'matched_...","[(Art. 973 Abs. 1 ZGB, 1), (Art. 848 ZGB, 1), ..."
7,8,eintragung erworbenen dinglichen rechte ansprü...,3,4,4,0.2727,"[Art. 973 ZGB, Art. 974 Abs. 2 ZGB, Art. 975 ZGB]","[Art. 975 Abs. 1 ZGB, Art. 974 Abs. 2 ZGB, Art...","[{'citation': 'Art. 975 Abs. 1 ZGB', 'matched_...","[(Art. 975 Abs. 1 ZGB, 1), (Art. 974 Abs. 2 ZG..."
8,8,erworbenen dinglichen rechte ansprüche schaden...,3,4,4,0.2727,"[Art. 974 Abs. 2 ZGB, Art. 975 ZGB, Art. 976a ...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_...","[(Art. 974 Abs. 2 ZGB, 1), (Art. 974 Abs. 1 ZG..."
9,8,dinglichen rechte ansprüche schadenersatz unge...,3,4,4,0.2727,"[Art. 974 Abs. 2 ZGB, Art. 975 ZGB, Art. 976a ...","[Art. 974 Abs. 2 ZGB, Art. 975 Abs. 1 ZGB, Art...","[{'citation': 'Art. 974 Abs. 2 ZGB', 'matched_...","[(Art. 974 Abs. 2 ZGB, 1), (Art. 975 Abs. 1 ZG..."



Done. DataFrames available: ngram_detailed_df, ngram_summary_df, best_per_n_df


In [210]:
# Build finetuning dataset from TOP-3 n-grams per train row.

# For each train row:

# 1) Expand gold citations using same logic as return_matching_citation_from_laws_de

# 2) Build n-grams from matched gold-law text

# 3) Retrieve top-k for each n-gram

# 4) Keep top-3 n-grams by gold-hit strength

# 5) Save dataset with gold-hit ratios



import json

import re

import time

from pathlib import Path

from collections import Counter, defaultdict



# -------------------------------

# 1) CONFIG

# -------------------------------

NGRAM_SIZES = [7, 8]

TOP_K_PER_NGRAM = 100

MIN_TOKEN_LEN = 2

MAX_NGRAMS_PER_N = 80         # set None for exhaustive per n

MAX_TRAIN_ROWS = None         # None => all rows in train_csv

PROGRESS_EVERY = 20

SAVE_EVERY = 50

PRINT_PER_ROW = True          # print row-level summary with top-3 n-grams

PRINT_PER_N = True            # print per-n evaluation progress inside each row



OUTPUT_PER_ROW_PATH = Path("../data/ngram_top3_per_row_dataset.json")

OUTPUT_FLAT_PATH = Path("../data/ngram_top3_flat_finetune_dataset.json")



# -------------------------------

# 2) PRECHECKS

# -------------------------------

if "train_csv" not in globals() or "laws_de" not in globals() or "vectorstore" not in globals():

    raise ValueError("train_csv, laws_de, and vectorstore must be loaded before running this cell.")



if "query" not in train_csv.columns or "gold_citations" not in train_csv.columns:

    raise ValueError("train_csv must contain 'query' and 'gold_citations' columns.")



# Ensure helpers exist (fallbacks if not already defined above).

if "_contains_token_safely" not in globals():

    def _contains_token_safely(token: str, text: str) -> bool:

        token = token.strip()

        if not token:

            return True

        pattern = rf"(?<![0-9A-Za-z]){re.escape(token)}(?![0-9A-Za-z])"

        return re.search(pattern, text, flags=re.IGNORECASE) is not None



if "_extract_article_number" not in globals():

    def _extract_article_number(citation_text: str) -> str | None:

        m = re.search(r"\bArt\.\s*([0-9]+[a-zA-Z]*)\b", str(citation_text), flags=re.IGNORECASE)

        return m.group(1).lower() if m else None



if "GERMAN_STOPWORDS" in globals() and isinstance(GERMAN_STOPWORDS, set):

    german_stopwords = GERMAN_STOPWORDS

else:

    german_stopwords = {

        "aber", "als", "am", "an", "auch", "auf", "aus", "bei", "bis", "da", "das", "dass", "dem", "den",

        "der", "des", "die", "doch", "du", "durch", "ein", "eine", "einer", "eines", "er", "es", "für",

        "hat", "hier", "ich", "ihr", "im", "in", "ist", "ja", "mit", "nach", "nicht", "oder", "sein",

        "sich", "sie", "sind", "so", "und", "vom", "von", "vor", "war", "was", "wenn", "wer", "wie", "wir",

        "wird", "wo", "zu", "zum", "zur", "über"

    }



# -------------------------------

# 3) PREPARE LAWS LOOKUP INDEX

# -------------------------------

laws_lookup = laws_de[["citation", "text"]].copy()

laws_lookup = laws_lookup.dropna(subset=["citation", "text"]).reset_index(drop=True)

laws_lookup["citation"] = laws_lookup["citation"].astype(str).str.strip()

laws_lookup["citation_cf"] = laws_lookup["citation"].str.casefold()

laws_lookup["text"] = laws_lookup["text"].astype(str)

laws_lookup["article_num"] = laws_lookup["citation"].apply(_extract_article_number)



# Exact lookup: citation_casefold -> row indices

exact_index = defaultdict(list)

for i, cf in enumerate(laws_lookup["citation_cf"]):

    exact_index[cf].append(i)



# Article lookup: article_num -> row indices

article_index = defaultdict(list)

for i, art in enumerate(laws_lookup["article_num"]):

    if art is not None:

        article_index[art].append(i)



citation_text_lookup_cf = {}

for _, r in laws_lookup.iterrows():

    key = r["citation_cf"]

    if key not in citation_text_lookup_cf:

        citation_text_lookup_cf[key] = r["text"]



# -------------------------------

# 4) HELPERS

# -------------------------------

def _matching_law_indices_for_citation(citation: str) -> list[int]:

    """Use the same matching strategy as return_matching_citation_from_laws_de, but return row indices."""

    citation_clean = str(citation).strip()

    if not citation_clean:

        return []



    exact = exact_index.get(citation_clean.casefold(), [])

    if exact:

        return exact



    words = [w for w in citation_clean.split(" ") if w.strip()]

    target_article = _extract_article_number(citation_clean)

    remaining_tokens = [

        w for w in words

        if w.lower() not in {"art.", "art"} and (target_article is None or w.lower() != target_article)

    ]



    candidate_indices = article_index.get(target_article, []) if target_article is not None else []

    if not candidate_indices:

        return []



    matched = []

    for idx in candidate_indices:

        law_citation = laws_lookup.at[idx, "citation"]

        if all(_contains_token_safely(word, law_citation) for word in remaining_tokens):

            matched.append(idx)

    return matched





def _build_ranked_ngrams(tokens: list[str], n: int, max_count: int | None = None) -> list[str]:

    if len(tokens) < n:

        return []

    grams = [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

    ranked = [g for g, _ in Counter(grams).most_common()]

    if isinstance(max_count, int) and max_count > 0:

        return ranked[:max_count]

    return ranked





def _save_json(path: Path, data: list) -> None:

    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:

        json.dump(data, f, ensure_ascii=False, indent=2)



# -------------------------------

# 5) MAIN LOOP (ALL TRAIN ROWS)

# -------------------------------

row_indices = list(range(len(train_csv)))

if isinstance(MAX_TRAIN_ROWS, int) and MAX_TRAIN_ROWS > 0:

    row_indices = row_indices[:MAX_TRAIN_ROWS]



per_row_dataset = []

flat_finetune_dataset = []



processed_rows = 0

skipped_empty = 0

skipped_no_matches = 0

start_time = time.time()



print("\n" + "=" * 120)

print("BUILDING TOP-3 NGRAM FINETUNING DATASET")

print("=" * 120)

print(f"Rows to process          : {len(row_indices)}")

print(f"N-gram sizes             : {NGRAM_SIZES}")

print(f"Top-k retrieval          : {TOP_K_PER_NGRAM}")

print(f"Max n-grams per n        : {MAX_NGRAMS_PER_N}")

print(f"Output per-row JSON      : {OUTPUT_PER_ROW_PATH}")

print(f"Output flat JSON         : {OUTPUT_FLAT_PATH}")



for pos, train_idx in enumerate(row_indices, start=1):

    row_start = time.time()

    row = train_csv.iloc[train_idx]

    query_text = str(row["query"]).strip()

    gold_base = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]



    if PRINT_PER_ROW:

        print("\n" + "-" * 120)

        print(f"[ROW {pos}/{len(row_indices)}] train_idx={train_idx}")

        print(f"  query_chars={len(query_text)} | gold_base_count={len(gold_base)}")



    if not query_text or not gold_base:

        skipped_empty += 1

        if PRINT_PER_ROW:

            print("  skipped: empty query or empty gold citations")

        continue



    # Expand gold citations using the same matching logic.

    gold_variant_to_base = defaultdict(set)

    gold_variant_text_lookup = {}

    source_chunks = []

    unmatched_base = []



    for base_cit in gold_base:

        matched_indices = _matching_law_indices_for_citation(base_cit)

        if not matched_indices:

            unmatched_base.append(base_cit)

            continue



        for idx in matched_indices:

            var_cit = laws_lookup.at[idx, "citation"]

            var_text = laws_lookup.at[idx, "text"]

            key = var_cit.casefold()

            gold_variant_to_base[key].add(base_cit)

            if key not in gold_variant_text_lookup:

                gold_variant_text_lookup[key] = var_text

                source_chunks.append(var_text)



    if PRINT_PER_ROW:

        print(

            f"  expanded_gold_variants={len(gold_variant_to_base)} | "

            f"unmatched_base={len(unmatched_base)}"

        )



    if not source_chunks:

        skipped_no_matches += 1

        if PRINT_PER_ROW:

            print("  skipped: no matched source law text chunks")

        continue



    source_text = "\n\n".join(source_chunks)

    clean = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", " ", source_text.lower())

    tokens = [t for t in clean.split() if t and len(t) >= MIN_TOKEN_LEN and t not in german_stopwords]



    if PRINT_PER_ROW:

        print(f"  source_text_chars={len(source_text)} | token_count={len(tokens)}")



    if not tokens:

        skipped_no_matches += 1

        if PRINT_PER_ROW:

            print("  skipped: no tokens after cleanup")

        continue



    # Evaluate all candidate n-grams for this row.

    eval_rows = []

    for n in NGRAM_SIZES:

        grams = _build_ranked_ngrams(tokens, n, MAX_NGRAMS_PER_N)

        if PRINT_PER_N:

            print(f"  evaluating n={n} with {len(grams)} n-grams")



        for g_idx, gram in enumerate(grams, start=1):

            docs = vectorstore.similarity_search(gram, k=TOP_K_PER_NGRAM)

            retrieved = [

                str(d.metadata.get("citation")).strip()

                for d in docs

                if d.metadata.get("citation") and str(d.metadata.get("citation")).strip()

            ]

            unique_retrieved = list(dict.fromkeys(retrieved))



            gold_hit_details = []

            for rc in unique_retrieved:

                key = rc.casefold()

                if key in gold_variant_to_base:

                    gold_hit_details.append(

                        {

                            "citation": rc,

                            "matched_gold_base": sorted(gold_variant_to_base[key]),

                            "cit_text": gold_variant_text_lookup.get(key, citation_text_lookup_cf.get(key, "")),

                        }

                    )



            base_hit_set = set()

            for hit in gold_hit_details:

                for base_name in hit["matched_gold_base"]:

                    base_hit_set.add(base_name)



            gold_hits_unique_bases = len(base_hit_set)

            gold_hits_unique_variants = len(gold_hit_details)

            gold_hits_doc_level_topk = sum(1 for rc in retrieved if rc.casefold() in gold_variant_to_base)



            eval_rows.append(

                {

                    "n": n,

                    "ngram": gram,

                    "retrieved_docs": len(docs),

                    "unique_retrieved_citations": len(unique_retrieved),

                    "gold_hits_unique_bases": gold_hits_unique_bases,

                    "gold_hits_unique_variants": gold_hits_unique_variants,

                    "gold_hits_doc_level_topk": gold_hits_doc_level_topk,

                    "gold_hit_ratio_base": round(gold_hits_unique_bases / len(gold_base), 4) if gold_base else 0.0,

                    "gold_hit_ratio_topk_docs": round(gold_hits_doc_level_topk / TOP_K_PER_NGRAM, 4),

                    "gold_hit_base_names": sorted(base_hit_set),

                    "gold_hit_names": [h["citation"] for h in gold_hit_details],

                    "gold_hit_details": gold_hit_details,

                }

            )



            if PRINT_PER_N and g_idx % 20 == 0:

                print(f"    n={n}: evaluated {g_idx}/{len(grams)} n-grams")



    if not eval_rows:

        skipped_no_matches += 1

        if PRINT_PER_ROW:

            print("  skipped: no evaluated n-gram rows")

        continue



    top3 = sorted(

        eval_rows,

        key=lambda r: (

            r["gold_hits_unique_bases"],

            r["gold_hits_unique_variants"],

            r["gold_hits_doc_level_topk"],

            r["unique_retrieved_citations"],

        ),

        reverse=True,

    )[:3]



    if PRINT_PER_ROW:

        print("  top-3 n-grams (with gold-hit ratios):")

        for rank, item in enumerate(top3, start=1):

            print(

                f"    {rank}. n={item['n']} | {item['ngram']} | "

                f"gold_hit_ratio_base={item['gold_hit_ratio_base']:.4f} | "

                f"gold_hit_ratio_topk_docs={item['gold_hit_ratio_topk_docs']:.4f}"

            )



    # Per-row dataset record.

    per_row_dataset.append(

        {

            "train_idx": int(train_idx),

            "query": query_text,

            "gold_base_citations": gold_base,

            "unmatched_base_citations": unmatched_base,

            "top3_ngrams": top3,

        }

    )



    # Flat finetuning records (one row per top n-gram).

    for rank, item in enumerate(top3, start=1):

        flat_finetune_dataset.append(

            {

                "id": f"train-{train_idx}-ngram-top{rank}",

                "train_idx": int(train_idx),

                "input": {

                    "case_query": query_text,

                    "instruction": "Generate one German legal retrieval search query.",

                },

                "output": {

                    "search_query": item["ngram"],

                },

                "meta": {

                    "rank_within_row": rank,

                    "n": int(item["n"]),

                    "gold_hit_ratio_base": item["gold_hit_ratio_base"],

                    "gold_hit_ratio_topk_docs": item["gold_hit_ratio_topk_docs"],

                    "gold_hits_unique_bases": int(item["gold_hits_unique_bases"]),

                    "gold_hits_unique_variants": int(item["gold_hits_unique_variants"]),

                    "gold_hits_doc_level_topk": int(item["gold_hits_doc_level_topk"]),

                    "gold_hit_base_names": item["gold_hit_base_names"],

                    "gold_hit_names": item["gold_hit_names"],

                    "top_k": TOP_K_PER_NGRAM,

                },

            }

        )



    processed_rows += 1

    row_elapsed = time.time() - row_start

    if PRINT_PER_ROW:

        print(f"  row_complete in {row_elapsed:.1f}s")



    if pos % PROGRESS_EVERY == 0 or pos == len(row_indices):

        elapsed = time.time() - start_time

        print(

            f"[{pos}/{len(row_indices)}] processed={processed_rows}, "

            f"skipped_empty={skipped_empty}, skipped_no_matches={skipped_no_matches}, "

            f"elapsed={elapsed/60:.1f} min"

        )



    if pos % SAVE_EVERY == 0:

        _save_json(OUTPUT_PER_ROW_PATH, per_row_dataset)

        _save_json(OUTPUT_FLAT_PATH, flat_finetune_dataset)

        if PRINT_PER_ROW:

            print(f"  checkpoint saved at row {pos}")



# Final save

_save_json(OUTPUT_PER_ROW_PATH, per_row_dataset)

_save_json(OUTPUT_FLAT_PATH, flat_finetune_dataset)



elapsed_total = time.time() - start_time

print("\n" + "=" * 120)

print("DATASET BUILD COMPLETE")

print("=" * 120)

print(f"Processed rows             : {processed_rows}/{len(row_indices)}")

print(f"Skipped empty              : {skipped_empty}")

print(f"Skipped no matches/tokens  : {skipped_no_matches}")

print(f"Flat finetune records      : {len(flat_finetune_dataset)}")

print(f"Per-row output             : {OUTPUT_PER_ROW_PATH}")

print(f"Flat output                : {OUTPUT_FLAT_PATH}")

print(f"Total runtime              : {elapsed_total/60:.1f} min")



if flat_finetune_dataset:

    mean_top1_ratio = sum(

        row["top3_ngrams"][0]["gold_hit_ratio_base"]

        for row in per_row_dataset

        if row.get("top3_ngrams")

    ) / max(1, sum(1 for row in per_row_dataset if row.get("top3_ngrams")))

    print(f"Mean top-1 gold_hit_ratio_base: {mean_top1_ratio:.4f}")

    display(pd.DataFrame(flat_finetune_dataset).head(10))



BUILDING TOP-3 NGRAM FINETUNING DATASET
Rows to process          : 1139
N-gram sizes             : [7, 8]
Top-k retrieval          : 100
Max n-grams per n        : 80
Output per-row JSON      : ../data/ngram_top3_per_row_dataset.json
Output flat JSON         : ../data/ngram_top3_flat_finetune_dataset.json

------------------------------------------------------------------------------------------------------------------------
[ROW 1/1139] train_idx=0
  query_chars=1494 | gold_base_count=3
  expanded_gold_variants=2 | unmatched_base=1
  source_text_chars=447 | token_count=32
  evaluating n=7 with 26 n-grams
    n=7: evaluated 20/26 n-grams
  evaluating n=8 with 25 n-grams
    n=8: evaluated 20/25 n-grams
  top-3 n-grams (with gold-hit ratios):
    1. n=7 | bevor behörde planung errichtung änderung anlagen entscheidet | gold_hit_ratio_base=0.6667 | gold_hit_ratio_topk_docs=0.0200
    2. n=7 | behörde planung errichtung änderung anlagen entscheidet prüft | gold_hit_ratio_base=0.6667 | gol

,id,train_idx,input,output,meta
0,train-0-ngram-top1,0,{'case_query': 'Die A AG betreibt seit den 197...,{'search_query': 'bevor behörde planung errich...,"{'rank_within_row': 1, 'n': 7, 'gold_hit_ratio..."
1,train-0-ngram-top2,0,{'case_query': 'Die A AG betreibt seit den 197...,{'search_query': 'behörde planung errichtung ä...,"{'rank_within_row': 2, 'n': 7, 'gold_hit_ratio..."
2,train-0-ngram-top3,0,{'case_query': 'Die A AG betreibt seit den 197...,{'search_query': 'planung errichtung änderung ...,"{'rank_within_row': 3, 'n': 7, 'gold_hit_ratio..."
3,train-1-ngram-top1,1,{'case_query': 'Die Alpha Consulting AG kann n...,{'search_query': 'dinglichen rechte ansprüche ...,"{'rank_within_row': 1, 'n': 7, 'gold_hit_ratio..."
4,train-1-ngram-top2,1,{'case_query': 'Die Alpha Consulting AG kann n...,{'search_query': 'erfolgt gutem glauben einen ...,"{'rank_within_row': 2, 'n': 7, 'gold_hit_ratio..."
5,train-1-ngram-top3,1,{'case_query': 'Die Alpha Consulting AG kann n...,{'search_query': 'gutem glauben einen eintrag ...,"{'rank_within_row': 3, 'n': 7, 'gold_hit_ratio..."
6,train-2-ngram-top1,2,{'case_query': 'Das Kompetenzzentrum Völkerstr...,{'search_query': 'freispruch erlass verjährung...,"{'rank_within_row': 1, 'n': 8, 'gold_hit_ratio..."
7,train-2-ngram-top2,2,{'case_query': 'Das Kompetenzzentrum Völkerstr...,{'search_query': 'ausland hatte ziel täter ung...,"{'rank_within_row': 2, 'n': 8, 'gold_hit_ratio..."
8,train-2-ngram-top3,2,{'case_query': 'Das Kompetenzzentrum Völkerstr...,{'search_query': 'strafbar täter ausland tat z...,"{'rank_within_row': 3, 'n': 7, 'gold_hit_ratio..."
9,train-3-ngram-top1,3,{'case_query': 'Die Linzer Stadtbahn AG ('LiSA...,{'search_query': 'sitz schweiz hatte 159 schie...,"{'rank_within_row': 1, 'n': 7, 'gold_hit_ratio..."


### for training row 1

In [245]:
### finetuned llm generated queries
train_index = 3
#search_queries = ['anlagen arten beispielsweise anlagen berechnen anlagen elektrische', 'anlagen berechnen anlagen elektrische heizanlagen gasfeuerungen fernwärmeversorgungsanlagen', 'gasfeuerungen fernwärmeversorgungsanlagen alle anlagen welches abfallgas abfälle']
# search_queries = ['eintragung löschen ändern grundbuchamt verlangen errungenschaft grundbuchamt', 'löschen ändern grundbuchamt verlangen errungenschaft grundbuchamt errungenschaft', 'ändern löschen ändern grundbuchamt verlangen errungenschaft grundbuchamt errungenschaft errungenschaftsrecht']
# search_queries = ['beschwerde kann erhoben werden gegen bestimmungen grundbuchblatte', 'kann erhoben werden gegen bestimmungen grundbuchblatte grundstückes', 'erhoben werden gegen bestimmungen grundbuchblatte grundstückes einschliesslich eingetragenen']
# search_queries = ['kann vorschriften römer statuts anwendbar sein vorliegt', 'vorschriften römer statuts anwendbar sein vorliegt tat', 'römer statuts anwendbar sein vorliegt tat ausübung öffentlichrechtlichen']
search_queries = ['schiedsgericht entscheidet seine zuständigkeit anfechtbar schiedsordnung schweizerische', 'entscheidet seine zuständigkeit anfechtbar schiedsordnung schweizerische schiedsinstitution', 'seine zuständigkeit anfechtbar schiedsordnung schweizerische schiedsinstitution kann']

train_row = train_csv.iloc[train_index]
gold_citations = [c.strip() for c in str(train_row["gold_citations"]).split(";") if c.strip()]
print("gold citations:", gold_citations)
gold_casefold = set(c.casefold() for c in gold_citations)

total_gold_hits = []

total_unique_retrieved = set()

for query in search_queries:
    print(f"\nSearch query: '{query}'")
    docs = vectorstore.similarity_search(query, k=500)
    retrieved_citations = [
        str(d.metadata.get("citation")).strip()
        for d in docs
        if d.metadata.get("citation") and str(d.metadata.get("citation")).strip()
    ]
    print(f"Retrieved citations: {retrieved_citations}")

    gold_hits = [rc for rc in retrieved_citations if rc.casefold() in gold_casefold]
    print(f"Gold hits in retrieved: {gold_hits}")
    for hit in gold_hits:
        if hit not in total_gold_hits:
            total_gold_hits.append(hit)
    total_unique_retrieved.update(retrieved_citations)

print(f"\nTotal unique gold hits across all queries: {total_gold_hits.__len__()} out of {len(gold_citations)} gold citations")
print(f"Total unique retrieved citations across all queries: {len(total_unique_retrieved)}")

print(f"precision: {len(total_gold_hits) / len(total_unique_retrieved):.4f} | recall: {len(total_gold_hits) / len(gold_citations):.4f}")
print(f"f1 score: {2 * (len(total_gold_hits) / len(total_unique_retrieved)) * (len(total_gold_hits) / len(gold_citations)) / ((len(total_gold_hits) / len(total_unique_retrieved)) + (len(total_gold_hits) / len(gold_citations))):.4f}")

gold citations: ['Art. 176 Abs. 1 IPRG', 'Art. 186 Abs. 1 IPRG', 'Art. 186 Abs. 3 IPRG', 'Art. 190 Abs. 3 IPRG']

Search query: 'schiedsgericht entscheidet seine zuständigkeit anfechtbar schiedsordnung schweizerische'
Retrieved citations: ['Art. 186 Abs. 1 IPRG', 'Art. 98 Abs. 2 IPRG', 'Art. 64 Abs. 1 IPRG', 'Art. 7 IPRG', 'Art. 63 Abs. 1 IPRG', 'Art. 6 IPRG', 'Art. 62 Abs. 1 IPRG', 'Art. 5 Abs. 1bis IPRG', 'Art. 186 Abs. 1bis IPRG', 'Art. 7 Abs. 1 ZISG', 'Art. 61 IPRG', 'Art. 27c Abs. 4 946.231.116.9', 'Art. 3 IPRG', 'Art. 27e Abs. 3 946.231.116.9', 'Art. 186 Abs. 3 IPRG', 'Art. 59 IPRG', 'Art. 27 Abs. 2 MVG', 'Art. 57 Abs. 2 UVG', 'Art. 89 Abs. 2 KVG', 'Art. 197 Abs. 1 IPRG', 'Art. 34 Abs. 1 IPRG', 'Art. 356 Abs. 1 ZPO', 'Art. 85 Abs. 3 IPRG', 'Art. 87 IRSG', 'Art. 41 Abs. 2 IPRG', 'Art. 97 IRSG', 'Art. 7 Abs. 2 ZISG', 'Art. 72 Abs. 3 IPRG', 'Art. 8a Abs. 1 IPRG', 'Art. 1 Abs. 1 IPRG', 'Art. 45a Abs. 2 IPRG', 'Art. 48 Abs. 3 IPRG', 'Art. 115 Abs. 2 IPRG', 'Art. 27quinquies Abs. 3 IVG

#### for training row 2

In [222]:
### finetuned llm generated queries
search_queries = ['ausland beklagten person gefoltert anderen personen körper', 'person gefoltert anderen personen körper angebotenen freundschaft', 'einen ausland beklagten person gefoltert anderen personen körper angebotenen freundschaft gewalt']
train_index = 2

train_row = train_csv.iloc[train_index]
gold_citations = [c.strip() for c in str(train_row["gold_citations"]).split(";") if c.strip()]
print("gold citations:", gold_citations)
gold_casefold = set(c.casefold() for c in gold_citations)

for query in search_queries:
    print(f"\nSearch query: '{query}'")
    docs = vectorstore.similarity_search(query, k=1000)
    retrieved_citations = [
        str(d.metadata.get("citation")).strip()
        for d in docs
        if d.metadata.get("citation") and str(d.metadata.get("citation")).strip()
    ]
    print(f"Retrieved citations: {retrieved_citations}")

    gold_hits = [rc for rc in retrieved_citations if rc.casefold() in gold_casefold]
    print(f"Gold hits in retrieved: {gold_hits}")

gold citations: ['Art. 264m StGB']

Search query: 'ausland beklagten person gefoltert anderen personen körper'
Retrieved citations: ['Art. 61b Abs. 2 USG', 'Art. 61a Abs. 2 USG', 'Art. 45 Abs. 2 RLG', 'Art. 39 Abs. 2 BÜPF', 'Art. 159a Abs. 2 MStG', 'Art. 20 Abs. 3 SVAG', 'Art. 96 Abs. 5 MWSTG', 'Art. 13 Abs. 2 VSG', 'Art. 188 Abs. 2 StGB', 'Art. 262 Abs. 3 StGB', 'Art. 166 Abs. 1 StPO', 'Art. 373 Abs. 3 StPO', 'Art. 123 Abs. 3 StGB', 'Art. 271 Abs. 2 StGB', 'Art. 185 Abs. 2 StGB', 'Art. 151c Abs. 2 MStG', 'Art. 128a Abs. 2 MStG', 'Art. 123 Abs. 1 StGB', 'Art. 58 Abs. 2 StPO', 'Art. 285 Abs. 4 StGB', 'Art. 106 Abs. 2 AVIG', 'Art. 129a Abs. 3 SSG', 'Art. 173 Abs. 2 StGB', 'Art. 122 Abs. 1 MStG', 'Art. 216 Abs. 1 StPO', 'Art. 262 Abs. 1 StGB', 'Art. 150 Abs. 2 StGB', 'Art. 276 Abs. 2 StGB', 'Art. 98 Abs. 2 MStG', 'Art. 329 Abs. 5 StGB', 'Art. 66 Abs. 2 MStP', 'Art. 184 Abs. 3 StGB', 'Art. 151b Abs. 3 MStG', 'Art. 66 Abs. 2 VStrR', 'Art. 52 Abs. 2 MStP', 'Art. 145 Abs. 2 MStG', 'Art. 124 A